# NeuroAtlas — A RAG-Powered Knowledge Assistant for Mental, Neurodevelopmental, Neurological & Sleep Disorders
Domain: mental disorders, neurodevelopmental disorders, neurological/movement disorders, and
sleep disorders, sourced from WHO, NIMH, NINDS, NHLBI, NICHD, NIGMS, CDC and AASM 


## 2.1 Load & Inspect




In [1]:
import pymupdf
def pdf_to_markdown(pdf_path, markdown_path):

    markdown = []

    with pymupdf.open(pdf_path) as pdf:
        for page_number, page in enumerate(pdf, start=1):
            markdown.append(f"## Page {page_number}\n\n")
            markdown.append(page.get_text())
            markdown.append("\n\n")

    # Creates/opens the output Markdown file in write mode using UTF-8 encoding.
    with open(markdown_path, "w", encoding="utf-8") as file:
        file.write("".join(markdown)) #one string

    print(f"Markdown file created: {markdown_path}")


pdf_to_markdown(
    "../data/documents/Neuro_dataset.pdf",
    "../data/documents/output.md"
)



Markdown file created: ../data/documents/output.md


# 2.2 Chunking Strategy

cleaning text


In [1]:
def clean_text(text):

    # Remove page number at the beginning
    text = re.sub(r'^\d+\s*\n', '', text)

    # Fix words broken by a hyphen and spaces
    text = re.sub(r'(\w)-\s+(\w)', r'\1\2', text)

    # Replace tabs/spaces after bullet points with one space
    text = re.sub(r'•\s*', '• ', text)

     # Remove reference/bibliography sections
    text = re.sub(
        r'(?im)^\s*(REFERENCES|BIBLIOGRAPHY)\s*$.*',
        '',
        text,
        flags=re.DOTALL
    )

    # Replace line breaks that occur inside a sentence/paragraph with a space.
    # Keep the line break when the next line starts with a bullet.
    text = re.sub(r'\n(?!•)', ' ', text)

    # Remove repeated PDF footer text
    text = re.sub(r'Copyright © National Academy of Sciences\.', '', text)
    text = re.sub(r'All rights reserved\.', '', text)
    text = re.sub(r'http://www\.nap\.edu/catalog/11617\.html', '', text)

    # Remove extra spaces
    text = re.sub(r'[ \t]+', ' ', text)

    # Clean up spaces around paragraphs
    text = re.sub(r' {2,}', ' ', text)

    return text.strip()

This cell imports the required libraries, defines the page ranges for each source document, provides a function to identify the source book for each page, and loads the extracted Markdown text for the RAG pipeline.

In [2]:
import re
from pathlib import Path
from semantic_chunkers import StatisticalChunker
from semantic_router.encoders import HuggingFaceEncoder

# Book page ranges
BOOK_RANGES = [
    ("WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders)", 1, 852),
    ("Fundamentals of Psychological Disorders (mental use)", 853, 1114),
    ("Handbook of Neurodevelopmental and Genetic Disorders in Adults (developmental)", 1115, 1622),
    ("Developmental Screening - CDC", 1623, 1641),
    ("Neurological Disorders - Public Health Challenges (neuro, WHO)", 1642, 1873),
    ("Global Status Report on Neurology (neuro 2, WHO)", 1874, 2158),
    ("Sleep Disorders and Sleep Deprivation (National Academies)", 2159, 2583),
    ("Rhythmic Movement Disorder case report (rmd, JCSM)", 2584, 2587),
]

# book names
def get_book_name(page_number):
    """Return the book name corresponding to a PDF page number."""
    for book_name, start_page, end_page in BOOK_RANGES:
        if start_page <= page_number <= end_page:
            return book_name
    return None

MARKDOWN_PATH = Path("../data/documents/output.md")
text = MARKDOWN_PATH.read_text(encoding="utf-8")

print(f"Text length: {len(text):,} characters")


c:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\.venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Text length: 7,202,013 characters


This cell splits the Markdown into individual pages, groups consecutive pages by source book, and displays the resulting page and book counts.

In [91]:
def split_into_pages(text):
    """Split Markdown text into individual pages."""
    pattern = r"^##\s*Page\s+(\d+)\s*$"
    parts = re.split(pattern, text, flags=re.MULTILINE)

    pages = []
    for i in range(1, len(parts), 2):
        page_number = int(parts[i])
        page_text = parts[i + 1].strip()
        if page_text:
            pages.append({"page_number": page_number, "text": page_text})
    return pages


def group_pages_by_book(pages):
    """Group consecutive pages that belong to the same book."""
    book_groups = []
    current_book = None
    current_pages = []

    for page in pages:
        book_name = get_book_name(page["page_number"])

        if book_name != current_book:
            if current_pages:
                book_groups.append({"book": current_book, "pages": current_pages})
            current_book = book_name
            current_pages = []

        current_pages.append(page)

    if current_pages:
        book_groups.append({"book": current_book, "pages": current_pages})

    return book_groups


pages = split_into_pages(text)
book_groups = group_pages_by_book(pages)

print(f"Number of pages: {len(pages)}")
print(f"Number of book groups: {len(book_groups)}")
for group in book_groups:
    print(f"{group['book']}: pages {group['pages'][0]['page_number']}-{group['pages'][-1]['page_number']}")


Number of pages: 2581
Number of book groups: 8
WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders): pages 1-852
Fundamentals of Psychological Disorders (mental use): pages 853-1114
Handbook of Neurodevelopmental and Genetic Disorders in Adults (developmental): pages 1116-1622
Developmental Screening - CDC: pages 1623-1641
Neurological Disorders - Public Health Challenges (neuro, WHO): pages 1642-1871
Global Status Report on Neurology (neuro 2, WHO): pages 1874-2158
Sleep Disorders and Sleep Deprivation (National Academies): pages 2159-2583
Rhythmic Movement Disorder case report (rmd, JCSM): pages 2584-2587


## Semantic Chunking Setup and Smoke Test

This cell initializes the embedding model and the semantic chunking strategy used to divide the document collection into meaningful sections.

The `HuggingFaceEncoder` uses the `sentence-transformers/all-MiniLM-L6-v2` model to convert text into embeddings. These embeddings allow the `StatisticalChunker` to measure semantic similarity between neighboring parts of the document and identify appropriate chunk boundaries.

The `StatisticalChunker` is configured with a minimum split size of 100 tokens, a maximum split size of 400 tokens, and a window size of 5. The minimum and maximum token limits control the approximate size of the semantic chunks, while the window size determines how neighboring text is considered when detecting semantic changes.

An additional similarity-based merge function, `merge_similar_neighbors()`, is used during experimentation to examine whether adjacent splits are still semantically similar and can safely be combined. Bullet points are protected from automatic merging so that separate list items are not incorrectly joined.

Before applying the strategy to the complete document collection, a smoke test is performed on page 1319. The page is cleaned, passed through the semantic chunker, and its top-level chunks and internal splits are displayed. This allows the chunking behavior to be inspected manually and helps verify that the selected parameters produce meaningful sections before processing the entire dataset.

In [4]:
from semantic_chunkers import ConsecutiveChunker
from semantic_chunkers import CumulativeChunker
import numpy as np


encoder = HuggingFaceEncoder(
    name="sentence-transformers/all-MiniLM-L6-v2"
)

chunker = StatisticalChunker(
    encoder=encoder,
    min_split_tokens=100,  # 100
    max_split_tokens=400,  # 400
    window_size=5
)


# chunker = ConsecutiveChunker(
#     encoder=encoder,
#     score_threshold=0.3
# )

# chunker = CumulativeChunker(
#     encoder=encoder,
#     score_threshold=0.3
# )


# 4. MERGE PASS — re-check adjacent splits, glue back together if still similar
def merge_similar_neighbors(splits, encoder, similarity_threshold=0.6):
    if len(splits) < 2:
        return splits

    embeddings = encoder(
        docs=splits,
        normalize_embeddings=True
    )

    merged = [splits[0]]

    for i in range(1, len(splits)):
        text = splits[i]

        # Don't merge if this starts a new bullet
        if text.startswith("•"):
            merged.append(text)
            continue

        similarity = float(
            np.dot(embeddings[i - 1], embeddings[i])
        )

        if similarity >= similarity_threshold:
            merged[-1] = merged[-1] + " " + text
        else:
            merged.append(text)

    return merged


# # Smoke test on page 1319
# # 3. Get page 1319
# test_page = next(
#     page for page in pages
#     if page["page_number"] == 1319
# )


# # 4. Clean the page
# clean_page_text = clean_text(test_page["text"])


# # 5. Chunk the cleaned page
# test_chunks = chunker([clean_page_text])


# # 6. Get the Chunk object
# chunk = test_chunks[0][0]


# # 7. Display the results
# print(f"Page: {test_page['page_number']}")
# print(f"Number of top-level chunks: {len(test_chunks[0])}")


# for c_idx, chunk in enumerate(test_chunks[0]):

#     print(
#         f"\n===== Chunk {c_idx + 1} "
#         f"({len(chunk.splits)} splits) ====="
#     )

#     for i, split in enumerate(chunk.splits):
#         print(f"\n--- Split {i + 1} ---")
#         print(split)

#     final_splits = merge_similar_neighbors(
#         chunk.splits,
#         encoder,
#         similarity_threshold=0.6
#     )

#     print(
#         f"\n{len(chunk.splits)} splits "
#         f"-> {len(final_splits)} after merge"
#     )

#     for i, s in enumerate(final_splits):
#         print(f"\n--- Merged split {i + 1} ---")
#         print(s)

In [5]:
# ============================================================
# LIBRARY + PIPELINE COMPATIBILITY TEST
# ============================================================

import sys
import importlib.metadata

print("=" * 70)
print("1. PYTHON ENVIRONMENT")
print("=" * 70)

print("Python executable:")
print(sys.executable)
print()

print("Python version:")
print(sys.version)


# ------------------------------------------------------------
# 2. CHECK LIBRARY VERSIONS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("2. LIBRARY VERSIONS")
print("=" * 70)

packages = [
    "fastapi",
    "uvicorn",
    "pydantic",
    "pydantic-settings",
    "ollama",
    "chromadb",
    "sentence-transformers",
    "semantic-router",
    "ultralytics",
    "pillow",
    "python-multipart",
    "python-dotenv",
    "pytest",
    "httpx",
]

for package in packages:
    try:
        print(
            f"{package:25} "
            f"{importlib.metadata.version(package)}"
        )
    except importlib.metadata.PackageNotFoundError:
        print(
            f"{package:25} NOT INSTALLED"
        )


# ------------------------------------------------------------
# 3. TEST IMPORTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("3. IMPORT TEST")
print("=" * 70)

try:
    import chromadb
    print("✓ chromadb")
except Exception as e:
    print("✗ chromadb:", e)

try:
    from sentence_transformers import SentenceTransformer
    print("✓ sentence-transformers")
except Exception as e:
    print("✗ sentence-transformers:", e)

try:
    from semantic_router.encoders import HuggingFaceEncoder
    print("✓ semantic-router")
except Exception as e:
    print("✗ semantic-router:", e)

try:
    from ultralytics import YOLO
    print("✓ ultralytics")
except Exception as e:
    print("✗ ultralytics:", e)

try:
    import ollama
    print("✓ ollama")
except Exception as e:
    print("✗ ollama:", e)


# ------------------------------------------------------------
# 4. TEST EMBEDDING MODEL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("4. EMBEDDING MODEL TEST")
print("=" * 70)

try:
    from semantic_router.encoders import HuggingFaceEncoder

    encoder = HuggingFaceEncoder(
        name="sentence-transformers/all-MiniLM-L6-v2"
    )

    test_embedding = encoder(
        docs=["This is a test sentence."]
    )

    print("✓ Embedding model works")
    print("Embedding type:", type(test_embedding))
    print("Embedding shape:", test_embedding.shape)

except Exception as e:
    print("✗ Embedding test failed:")
    print(type(e).__name__, e)


# ------------------------------------------------------------
# 5. TEST CHROMA VECTOR STORE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("5. CHROMA VECTOR STORE TEST")
print("=" * 70)

try:
    client = chromadb.PersistentClient(
        path="../data/vector_store"
    )

    collection = client.get_collection(
        name="neurohealth_chunks"
    )

    print("✓ Chroma vector store loaded")
    print("Collection:", collection.name)
    print("Number of chunks:", collection.count())

except Exception as e:
    print("✗ Chroma test failed:")
    print(type(e).__name__, e)


# ------------------------------------------------------------
# 6. TEST CHROMA RETRIEVAL
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("6. RETRIEVAL TEST")
print("=" * 70)

try:
    query = "What are the symptoms of depression?"

    query_embedding = encoder(
        docs=[query],
        normalize_embeddings=True
    )

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=5
    )

    print("✓ Chroma retrieval works")
    print("Number of results:", len(results["documents"][0]))

    print("\nFirst retrieved document:")
    print(results["documents"][0][0][:500])

except Exception as e:
    print("✗ Retrieval test failed:")
    print(type(e).__name__, e)


# ------------------------------------------------------------
# 7. TEST YOLO MODELS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("7. YOLO TEST")
print("=" * 70)

try:
    from ultralytics import YOLO

    yolo_paths = {
        "down_syndrome":
            "../runs/classify/runs/down_syndrome_cls-5/weights/best.pt",

        "autism":
            "../runs/classify/runs/autism_cls-4/weights/best.pt",

        "depression":
            "../runs/classify/runs/depression_cls-4/weights/best.pt",
    }

    yolo_models = {}

    for name, path in yolo_paths.items():

        print(f"\nLoading {name}...")

        model = YOLO(path)

        yolo_models[name] = model

        print(f"✓ {name} model loaded")

except Exception as e:
    print("✗ YOLO test failed:")
    print(type(e).__name__, e)


# ------------------------------------------------------------
# 8. TEST OLLAMA CONNECTION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("8. OLLAMA TEST")
print("=" * 70)

try:
    import ollama

    response = ollama.chat(
        model="llama3.2:latest",
        messages=[
            {
                "role": "user",
                "content": "Reply with exactly: TEST OK"
            }
        ],
        options={
            "temperature": 0,
            "num_ctx": 4096,
        },
        keep_alive=0,
    )

    answer = response["message"]["content"]

    print("✓ Ollama works")
    print("Response:", answer)

except Exception as e:
    print("✗ Ollama test failed:")
    print(type(e).__name__, e)


print("\n" + "=" * 70)
print("TEST COMPLETE")
print("=" * 70)

1. PYTHON ENVIRONMENT
Python executable:
c:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\.venv312\Scripts\python.exe

Python version:
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]

2. LIBRARY VERSIONS
fastapi                   0.115.0
uvicorn                   0.52.4
pydantic                  2.13.5
pydantic-settings         2.15.0
ollama                    0.3.3
chromadb                  1.5.9
sentence-transformers     3.1.1
semantic-router           0.1.16
ultralytics               8.4.146
pillow                    10.4.0
python-multipart          0.0.32
python-dotenv             1.2.3
pytest                    8.3.3
httpx                     0.28.1

3. IMPORT TEST
✓ chromadb
✓ sentence-transformers
✓ semantic-router
✓ ultralytics
✓ ollama

4. EMBEDDING MODEL TEST
✓ Embedding model works
Embedding type: <class 'list'>
✗ Embedding test failed:
AttributeError 'list' object has no attribute 'shape'

5. C

 Coordinate-Based Header/Footer and Reference Detection

This cell detects repeated headers and footers using their position on the original PDF pages rather than relying only on repeated extracted text. The detection is performed separately for each source book to reduce false matches between different documents.

The cell also identifies reference and bibliography pages using multiple citation patterns. Numbered questions, ordinary name lists, and clock-time values are excluded from the citation scoring to reduce false positives.

The detected running headers and footers are stored in `boilerplate`, while reference pages are marked later by `mark_reference_sections()` before the final RAG chunks are created.

In [90]:
"""
Coordinate-based header/footer removal
=======================================

Detects repeated headers and footers from their actual position
on the PDF page and detects reference/bibliography sections
using citation signals.

Requires the original PDF (Neuro_dataset.pdf).
"""

import re
import pymupdf as fitz
from collections import Counter

# ============================================================
# Reference-page detection
# ============================================================

def is_reference_page(text):
    """Return True if the page contains a References/Bibliography heading."""
    for line in text.splitlines():
        stripped = line.strip()
        stripped = stripped.rstrip(":").rstrip("0123456789")

        if re.fullmatch(r"references", stripped, re.I):
            return True

        if re.fullmatch(r"bibliography", stripped, re.I):
            return True

    return False

# ============================================================
# Header/footer normalization
# ============================================================

def normalize_block(text):
    """Normalize a PDF margin block for comparison."""
    lines = text.splitlines()

    lines = [
        line
        for line in lines
        if not re.fullmatch(r"\s*\d+\s*", line)
    ]

    return re.sub(r"\s+", " ", " ".join(lines)).strip()

# ============================================================
# Running header/footer detection
# ============================================================

def find_running_headers_footers(
    pdf_path,
    book_ranges,
    top_pct=0.12,
    bottom_pct=0.12,
    ratio_threshold=0.20,
    min_abs_count=8,
    min_book_pages_for_ratio=20,
):
    """
    Detect repeated text blocks in the top or bottom margin.
    Detection is performed separately for each book.
    """
    with fitz.open(pdf_path) as doc:
        def book_index(page_number):
            for i, (_, start_page, end_page) in enumerate(book_ranges):
                if start_page <= page_number <= end_page:
                    return i
            return -1

        margin_hits = [Counter() for _ in book_ranges]

        for pdf_page_num in range(1, len(doc) + 1):
            book_idx = book_index(pdf_page_num)

            if book_idx == -1:
                continue

            page = doc[pdf_page_num - 1]
            page_height = page.rect.height
            top_band = page_height * top_pct
            bottom_band = page_height * (1 - bottom_pct)
            seen_this_page = set()

            for block in page.get_text("blocks"):
                x0, y0, x1, y1, text = block[:5]
                normalized = normalize_block(text)

                if not normalized:
                    continue

                if y1 <= top_band or y0 >= bottom_band:
                    seen_this_page.add(normalized)

            for text in seen_this_page:
                margin_hits[book_idx][text] += 1

        boilerplate = set()
        skipped_books = []

        for book_idx, (name, start_page, end_page) in enumerate(book_ranges):
            number_of_pages = end_page - start_page + 1

            if number_of_pages < min_book_pages_for_ratio:
                skipped_books.append((name, number_of_pages))
                continue

            for text, count in margin_hits[book_idx].items():
                ratio = count / number_of_pages

                if ratio >= ratio_threshold and count >= min_abs_count:
                    boilerplate.add(text)

    if skipped_books:
        print("Skipped (too small for ratio detection, check manually):")
        for book in skipped_books:
            print(" -", book)

    return boilerplate

# ============================================================
# Reference citation patterns
# ============================================================

NUMBERED_REF = re.compile(
    r"(?:^|\n)\s*\d{1,3}\.\s"
)

JOURNAL_CITATION = re.compile(
    r"\b\d{1,4}"
    r"(?:\(\w+\.?\s?\d*\))?"
    r"\s*:\s*"
    r"(?!00\b)"
    r"[SsA-Z]?\d{1,5}"
)

APA_AUTHOR = re.compile(
    r"\b[A-Z][a-zA-Z\-]+,\s+[A-Z]\."
)

YEAR_PAREN = re.compile(
    r"\(\d{4}[a-z]?\)"
)

VANCOUVER_AUTHOR = re.compile(
    r"\b[A-Z][a-zA-Z\-]+\s[A-Z]{1,3}\b[,.]"
)

# ============================================================
# Rejoin wrapped lines
# ============================================================

def rejoin_wrapped_lines(text):
    """Join hard-wrapped lines into continuous text."""
    lines = text.splitlines()
    output = []
    buffer = []

    for line in lines:
        stripped = line.strip()

        if not stripped:
            continue

        buffer.append(stripped)

        if stripped.endswith((".", "!", "?", ":")):
            output.append(" ".join(buffer))
            buffer = []

    if buffer:
        output.append(" ".join(buffer))

    return "\n".join(output)

# ============================================================
# Count numbered reference-style lines
# ============================================================

def count_numbered_refs(joined_text):
    """
    Count numbered reference-style lines while ignoring
    numbered questions.
    """
    count = 0

    for line in joined_text.splitlines():
        stripped = line.strip()

        if not re.match(r"\d{1,3}\.\s", stripped):
            continue

        if stripped.endswith("?"):
            continue

        count += 1

    return count

# ============================================================
# Count author-style citation signals
# ============================================================

def count_author_signals(joined_text):
    """
    Count author patterns only when the same line also contains
    a year or journal-style citation.
    """
    count = 0

    for line in joined_text.splitlines():
        has_author = (
            bool(APA_AUTHOR.search(line))
            or bool(VANCOUVER_AUTHOR.search(line))
        )

        has_year_or_journal = (
            bool(YEAR_PAREN.search(line))
            or bool(JOURNAL_CITATION.search(line))
        )

        if has_author and has_year_or_journal:
            count += 1

    return count

# ============================================================
# Reference citation score
# ============================================================

def reference_signal_score(text):
    """
    Calculate citation-shaped signals per 100 words.
    """
    joined = rejoin_wrapped_lines(text)
    word_count = max(len(joined.split()), 1)
    signals = 0

    signals += count_numbered_refs(joined)
    signals += len(JOURNAL_CITATION.findall(joined))
    signals += len(YEAR_PAREN.findall(joined))
    signals += count_author_signals(joined)

    return signals / (word_count / 100)

# ============================================================
# Mark reference sections
# ============================================================

def mark_reference_sections(
    book_pages,
    score_threshold=4.0,
    continuation_threshold=3.0
):
    """
    Mark reference and bibliography pages within one book.

    A reference section starts when an explicit reference heading
    is found or the score reaches score_threshold.

    Once a reference section has started, continuation pages remain
    marked when their score reaches continuation_threshold.
    """
    in_reference_section = False
    marked = []

    for page in book_pages:
        text = page["text"]
        heading_hit = is_reference_page(text)
        score = reference_signal_score(text)

        # Explicit reference/bibliography heading
        if heading_hit:
            in_reference_section = True
            is_ref = True

        # Continuation of an existing reference section
        elif (
            in_reference_section
            and score >= continuation_threshold
        ):
            is_ref = True

        # Start a new reference section
        elif (
            not in_reference_section
            and score >= score_threshold
        ):
            in_reference_section = True
            is_ref = True

        # Normal prose
        else:
            in_reference_section = False
            is_ref = False

        marked.append({
            **page,
            "is_reference": is_ref,
            "_ref_score": round(score, 1)
        })

    return marked

# ============================================================
# Remove detected boilerplate
# ============================================================

def strip_boilerplate(text, boilerplate):
    """Remove lines matching detected running headers/footers."""
    def normalize_line(line):
        return re.sub(r"\s+", " ", line.strip())

    lines = text.splitlines()

    kept = [
        line
        for line in lines
        if normalize_line(line) not in boilerplate
    ]

    return "\n".join(kept)

# ============================================================
# Detect running headers and footers
# ============================================================

boilerplate = find_running_headers_footers(
    "../data/documents/Neuro_dataset.pdf",
    BOOK_RANGES
)

print(
    f"\nDetected {len(boilerplate)} "
    f"true running headers/footers:"
)

for item in sorted(boilerplate):
    print(" -", item)

Skipped (too small for ratio detection, check manually):
 - ('Developmental Screening - CDC', 19)
 - ('Rhythmic Movement Disorder case report (rmd, JCSM)', 4)

Detected 8 true running headers/footers:
 - Clinical Descriptions and Diagnostic Requirements for ICD-11 Mental, Behavioural or Neurodevelopmental Disorders
 - Copyright © National Academy of Sciences. All rights reserved.
 - DISORDERS WITH BROADER-SPECTRUM EFFECTS
 - Global status report on neurology
 - Neurological disorders: public health challenges
 - SLEEP DISORDERS AND SLEEP DEPRIVATION
 - Sleep Disorders and Sleep Deprivation: An Unmet Public Health Problem http://www.nap.edu/catalog/11617.html
 - neurological disorders: a public health approach


 Create Final RAG Chunks

This cell processes all source books, cleans their pages, applies semantic chunking, preserves page positions, and assigns each final chunk its source book, page range, and token count. The resulting chunks are stored in `all_chunks` for the next processing stages.

In [7]:
### Create Final RAG Chunks

all_chunks = []


def combine_pages_with_offsets(pages):
    """Combine pages while preserving their character positions."""

    combined_text = ""
    page_offsets = []

    for page in pages:

        start = len(combined_text)

        combined_text += page["text"]

        end = len(combined_text)

        page_offsets.append({
            "page_number": page["page_number"],
            "start": start,
            "end": end
        })

        combined_text += "\n\n"

    return combined_text, page_offsets


def get_page_range(start_pos, end_pos, page_offsets):
    """Find the first and last page covered by a text span."""

    page_start = None
    page_end = None

    for page in page_offsets:

        if (
            page_start is None
            and start_pos < page["end"]
        ):
            page_start = page["page_number"]

        if end_pos <= page["end"]:

            page_end = page["page_number"]

            break

    return page_start, page_end


# ============================================================
# Pages removed as reference/bibliography pages
# ============================================================

reference_pages_removed = []


# ============================================================
# Process every book
# ============================================================

for group in book_groups:

    book_name = group["book"]
    book_pages = group["pages"]

    print(
        f"\nProcessing: {book_name}"
    )


    # --------------------------------------------------------
    # Mark reference pages
    # --------------------------------------------------------

    marked_pages = mark_reference_sections(
        book_pages
    )


    # --------------------------------------------------------
    # Clean pages
    # --------------------------------------------------------

    cleaned_pages = []


    for page in marked_pages:

        # Remove reference/bibliography pages
        if page["is_reference"]:

            reference_pages_removed.append(
                page["page_number"]
            )

            continue


        # ----------------------------------------------------
        # Remove detected running headers and footers
        # ----------------------------------------------------

        stripped_text = strip_boilerplate(
            page["text"],
            boilerplate
        )


        # ----------------------------------------------------
        # Apply normal text cleaning
        # ----------------------------------------------------

        cleaned_text = clean_text(
            stripped_text
        )


        # ----------------------------------------------------
        # Skip empty pages
        # ----------------------------------------------------

        if not cleaned_text:
            continue


        # ----------------------------------------------------
        # Skip pages containing only a page number
        # ----------------------------------------------------

        if re.fullmatch(
            r"\d+",
            cleaned_text
        ):
            continue


        cleaned_pages.append({
            "page_number": page["page_number"],
            "text": cleaned_text
        })


    # --------------------------------------------------------
    # Combine pages while preserving positions
    # --------------------------------------------------------

    book_text, page_offsets = combine_pages_with_offsets(
        cleaned_pages
    )


    # --------------------------------------------------------
    # Semantic chunking
    # --------------------------------------------------------

    book_chunks = chunker([
        book_text
    ])


    book_chunk_count = 0
    search_position = 0


    # Each Chunk object represents one semantic chunk
    for chunk in book_chunks[0]:

        # Join the internal splits into the final chunk text
        chunk_text = " ".join(
            chunk.splits
        ).strip()


        # ----------------------------------------------------
        # Find the chunk in the combined book text
        # ----------------------------------------------------

        chunk_start = book_text.find(
            chunk.splits[0],
            search_position
        )


        if chunk_start == -1:

            raise ValueError(
                "Could not locate chunk in source text:\n"
                f"{chunk_text[:200]}"
            )


        chunk_end = (
            chunk_start
            + len(chunk_text)
        )


        # ----------------------------------------------------
        # Determine source page range
        # ----------------------------------------------------

        page_start, page_end = get_page_range(
            chunk_start,
            chunk_end,
            page_offsets
        )


        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        if (
            page_start in reference_pages_removed
            or page_end in reference_pages_removed
        ):

            raise ValueError(
                f"Reference page "
                f"{page_start}-{page_end} "
                f"unexpectedly reached chunking."
            )


        # ----------------------------------------------------
        # Save final RAG chunk
        # ----------------------------------------------------

        all_chunks.append({
            "text": chunk_text,
            "book": book_name,
            "page_start": page_start,
            "page_end": page_end,
            "token_count": chunk.token_count
        })


        # Continue searching after this chunk
        search_position = chunk_end

        book_chunk_count += 1


    print(
        f"Final chunks from this book: "
        f"{book_chunk_count}"
    )


# ============================================================
# Final checks
# ============================================================

print(
    "\n" + "=" * 60
)

print(
    "FINAL CHUNKING SUMMARY"
)

print(
    "=" * 60
)


print(
    f"Reference pages removed: "
    f"{len(reference_pages_removed):,}"
)


print(
    f"Total final chunks: "
    f"{len(all_chunks):,}"
)


print(
    "\nReference pages removed:"
)


print(
    sorted(reference_pages_removed)
)


Processing: WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders)


2026-09-12 04:25:23 INFO semantic_chunkers.utils.logger Single document exceeds the maximum token limit of 400. Splitting to sentences before semantically merging.
100%|██████████| 176/176 [00:13<00:00, 13.14it/s]


Final chunks from this book: 1581

Processing: Fundamentals of Psychological Disorders (mental use)


2026-09-12 04:25:38 INFO semantic_chunkers.utils.logger Single document exceeds the maximum token limit of 400. Splitting to sentences before semantically merging.
100%|██████████| 99/99 [00:06<00:00, 16.40it/s]


Final chunks from this book: 751

Processing: Handbook of Neurodevelopmental and Genetic Disorders in Adults (developmental)


...

Final chunks from this book: 444

Processing: Sleep Disorders and Sleep Deprivation (National Academies)


2026-09-12 04:26:06 INFO semantic_chunkers.utils.logger Single document exceeds the maximum token limit of 400. Splitting to sentences before semantically merging.
100%|██████████| 81/81 [00:05<00:00, 13.58it/s]
2026-09-12 04:26:12 INFO semantic_chunkers.utils.logger Single document exceeds the maximum token limit of 400. Splitting to sentences before semantically merging.


Final chunks from this book: 718

Processing: Rhythmic Movement Disorder case report (rmd, JCSM)


100%|██████████| 2/2 [00:00<00:00, 18.80it/s]

Final chunks from this book: 14

FINAL CHUNKING SUMMARY
Reference pages removed: 230
Total final chunks: 5,139

Reference pages removed:
[5, 34, 35, 36, 37, 1129, 1144, 1145, 1164, 1165, 1192, 1193, 1194, 1215, 1216, 1217, 1218, 1219, 1220, 1221, 1222, 1223, 1224, 1225, 1244, 1245, 1246, 1247, 1248, 1249, 1250, 1251, 1273, 1274, 1275, 1276, 1277, 1278, 1279, 1280, 1281, 1282, 1283, 1303, 1304, 1305, 1306, 1307, 1308, 1325, 1326, 1327, 1328, 1329, 1330, 1331, 1353, 1354, 1355, 1356, 1357, 1358, 1359, 1360, 1361, 1376, 1377, 1378, 1379, 1395, 1396, 1397, 1416, 1417, 1418, 1419, 1420, 1421, 1422, 1423, 1432, 1433, 1434, 1435, 1442, 1443, 1444, 1455, 1456, 1466, 1467, 1468, 1469, 1482, 1483, 1484, 1485, 1486, 1499, 1500, 1501, 1502, 1503, 1515, 1516, 1517, 1518, 1519, 1523, 1524, 1542, 1543, 1544, 1545, 1546, 1568, 1569, 1570, 1571, 1572, 1573, 1574, 1575, 1590, 1591, 1592, 1593, 1594, 1609, 1610, 1611, 1612, 1613, 1658, 1677, 1678, 1692, 1706, 1720, 1721, 1722, 1736, 1737, 1747, 1762, 176

In [8]:
test_pages = [
    829,
    917,
    1008,
    1274,
    1275,
    1721,
    1722,
    2322,
    2346,
    2347,
    2348,
    2349,
    2350,
    2351,
    2529
]

for group in book_groups:

    marked_pages = mark_reference_sections(
        group["pages"]
    )

    for page in marked_pages:

        if page["page_number"] in test_pages:

            print(
                f"Page {page['page_number']} | "
                f"Reference: {page['is_reference']} | "
                f"Score: {page['_ref_score']}"
            )

Page 829 | Reference: False | Score: 0.0
Page 917 | Reference: False | Score: 2.9
Page 1008 | Reference: False | Score: 1.0
Page 1274 | Reference: True | Score: 6.2
Page 1275 | Reference: True | Score: 5.5
Page 1721 | Reference: True | Score: 10.8
Page 1722 | Reference: True | Score: 4.5
Page 2322 | Reference: False | Score: 0.8
Page 2346 | Reference: True | Score: 5.6
Page 2347 | Reference: True | Score: 4.3
Page 2348 | Reference: True | Score: 3.4
Page 2349 | Reference: True | Score: 5.6
Page 2350 | Reference: True | Score: 5.0
Page 2351 | Reference: True | Score: 4.6
Page 2529 | Reference: False | Score: 0.0


In [9]:
bad_pages = [
    1721, 1722,
    1274, 1275,
    2346, 2347, 2348
]

for chunk in all_chunks:

    if chunk["page_start"] in bad_pages:

        print(
            f"Page {chunk['page_start']} still exists in all_chunks:"
        )

        print(chunk["text"][:300])

Safeguard for Oversized Chunks

This cell checks the semantic chunks and recursively splits any chunk larger than 600 tokens. Splits are made near sentence boundaries when possible, and token counts are recalculated for each resulting chunk. The final chunk count and token-size statistics are then displayed to verify the resulting RAG chunks.

In [10]:
import re
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")


def split_large_chunk(chunk, max_tokens=600):
    """Recursively split a chunk until every piece is <= max_tokens."""

    text = chunk["text"]

    token_count = len(enc.encode(text))

    # Already small enough
    if token_count <= max_tokens:
        return [{
            **chunk,
            "text": text,
            "token_count": token_count
        }]

    mid = len(text) // 2

    # Find sentence boundaries
    matches = list(
        re.finditer(r"[.!?]\s+", text)
    )

    if matches:
        split_point = min(
            matches,
            key=lambda m: abs(m.end() - mid)
        ).end()
    else:
        # Fallback to nearest space
        split_point = text.find(" ", mid)

        if split_point == -1:
            split_point = mid

    first_text = text[:split_point].strip()
    second_text = text[split_point:].strip()

    first_chunk = {
        **chunk,
        "text": first_text,
        "token_count": len(enc.encode(first_text))
    }

    second_chunk = {
        **chunk,
        "text": second_text,
        "token_count": len(enc.encode(second_text))
    }

    # Recursively split both pieces if necessary
    return (
        split_large_chunk(first_chunk, max_tokens)
        + split_large_chunk(second_chunk, max_tokens)
    )


final_rag_chunks = []

for chunk in all_chunks:
    final_rag_chunks.extend(
        split_large_chunk(chunk, max_tokens=600)
    )


print(f"Before safeguard: {len(all_chunks):,}")
print(f"After safeguard:  {len(final_rag_chunks):,}")


token_counts = [
    chunk["token_count"]
    for chunk in final_rag_chunks
]
print()
print(f"Total chunks: {len(final_rag_chunks):,}")
print(f"Average tokens: {sum(token_counts) / len(token_counts):.1f}")
print(f"Minimum tokens: {min(token_counts)}")
print(f"Maximum tokens: {max(token_counts)}")

Before safeguard: 5,139
After safeguard:  5,252

Total chunks: 5,252
Average tokens: 240.5
Minimum tokens: 22
Maximum tokens: 587


 Assign IDs and Save Semantic Chunks

This cell assigns a unique `chunk_id` to each final RAG chunk, summarizes the number of chunks generated from each source book, inspects the first five chunks and their metadata, and saves the complete chunk dataset as `semantic_chunks.json` for use in the embedding and vector-store stages.

In [11]:
from collections import Counter
import json
from pathlib import Path

for i, chunk in enumerate(final_rag_chunks):
    chunk["chunk_id"] = i

chunk_counts = Counter(chunk["book"] for chunk in final_rag_chunks)

for book, count in chunk_counts.items():
    print(f"{count:5d} chunks - {book}")

for chunk in final_rag_chunks[:5]:
    print("\n" + "=" * 80)
    print(f"Chunk ID:     {chunk['chunk_id']}")
    print(f"Book:         {chunk['book']}")
    print(f"Pages:        {chunk['page_start']}-{chunk['page_end']}")
    print(f"Token count:  {chunk['token_count']}")
    print(f"Text length:  {len(chunk['text'])}")
    print("\nText:")
    print(chunk["text"][:1000])

OUTPUT_PATH = Path("../data/documents/semantic_chunks.json")

with open(OUTPUT_PATH, "w", encoding="utf-8") as file:
    json.dump(final_rag_chunks, file, ensure_ascii=False, indent=2)

print(f"\nSemantic chunks saved to: {OUTPUT_PATH}")

 1593 chunks - WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders)
  760 chunks - Fundamentals of Psychological Disorders (mental use)
 1073 chunks - Handbook of Neurodevelopmental and Genetic Disorders in Adults (developmental)
   49 chunks - Developmental Screening - CDC
  593 chunks - Neurological Disorders - Public Health Challenges (neuro, WHO)
  447 chunks - Global Status Report on Neurology (neuro 2, WHO)
  723 chunks - Sleep Disorders and Sleep Deprivation (National Academies)
   14 chunks - Rhythmic Movement Disorder case report (rmd, JCSM)

Chunk ID:     0
Book:         WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders)
Pages:        1-4
Token count:  153
Text length:  663

Text:
Clinical descriptions and diagnostic requirements for ICD-11 mental, behavioural and neurodevelopmental disorders Clinical descriptions and diagnostic requirements for ICD-11 mental, behavioural and neurodevelopmental disorders Clinical descriptions and diagnost

## 2.3 Embeddings & Vector Store

In [12]:
import json
from pathlib import Path
import chromadb
import numpy as np


# Load final semantic chunks
INPUT_PATH = Path("../data/documents/semantic_chunks.json")

# open file in read mode
with open(INPUT_PATH, "r", encoding="utf-8") as file:
    final_rag_chunks = json.load(file)


print(f"Loaded {len(final_rag_chunks):,} chunks")


# Create persistent Chroma database
CHROMA_PATH = "../data/vector_store"

#Chroma client whose data is persisted on disk.
client = chromadb.PersistentClient( path=CHROMA_PATH)


COLLECTION_NAME = "neurohealth_chunks"


# Recreate collection when rerunning the notebook
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass


collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"description": "NeuroHealth RAG semantic chunks"}
)


# Generate and store embeddings in batches
BATCH_SIZE = 100

for start in range(0, len(final_rag_chunks), BATCH_SIZE):

    batch = final_rag_chunks[start:start + BATCH_SIZE]

    # Extract only the text from each chunk for embedding
    texts = [
        chunk["text"]
        for chunk in batch
    ]

    # every chunk's text is converted into a numerical vector.
    embeddings = encoder(
        docs=texts,
        normalize_embeddings=True
    )

    #Convert embeddings to normal Python lists
    embeddings = np.asarray(embeddings).tolist()

    #Every Chroma record needs a unique ID.
    ids = [
        str(chunk["chunk_id"])
        for chunk in batch
    ]

    #This stores the source information alongside every embedding.
    metadatas = [
        {
            "book": chunk["book"],
            "page_start": chunk["page_start"],
            "page_end": chunk["page_end"]
        }
        for chunk in batch
    ]

    # where the actual storage happens.
    collection.add(
        ids=ids,
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas
    )

    #SHOW progress
    print(
        f"Stored {min(start + BATCH_SIZE, len(final_rag_chunks)):,}"
        f"/{len(final_rag_chunks):,} chunks"
    )


print("\nVector store created successfully.")
print(f"Collection: {COLLECTION_NAME}")
print(f"Number of stored chunks: {collection.count():,}")
print(f"Persistent path: {CHROMA_PATH}")

Loaded 5,252 chunks
Stored 100/5,252 chunks
Stored 200/5,252 chunks
Stored 300/5,252 chunks
Stored 400/5,252 chunks
Stored 500/5,252 chunks
Stored 600/5,252 chunks
Stored 700/5,252 chunks
Stored 800/5,252 chunks
Stored 900/5,252 chunks
Stored 1,000/5,252 chunks
Stored 1,100/5,252 chunks
Stored 1,200/5,252 chunks
Stored 1,300/5,252 chunks
Stored 1,400/5,252 chunks
Stored 1,500/5,252 chunks
Stored 1,600/5,252 chunks
Stored 1,700/5,252 chunks
Stored 1,800/5,252 chunks
Stored 1,900/5,252 chunks
Stored 2,000/5,252 chunks
Stored 2,100/5,252 chunks
Stored 2,200/5,252 chunks
Stored 2,300/5,252 chunks
Stored 2,400/5,252 chunks
Stored 2,500/5,252 chunks
Stored 2,600/5,252 chunks
Stored 2,700/5,252 chunks
Stored 2,800/5,252 chunks
Stored 2,900/5,252 chunks
Stored 3,000/5,252 chunks
Stored 3,100/5,252 chunks
Stored 3,200/5,252 chunks
Stored 3,300/5,252 chunks
Stored 3,400/5,252 chunks
Stored 3,500/5,252 chunks
Stored 3,600/5,252 chunks
Stored 3,700/5,252 chunks
Stored 3,800/5,252 chunks
Stored 3,9

# Retrieval & Prompting

In [13]:
def retrieve_chunks(query, top_k=5):

    # Generate an embedding for the user's question
    query_embedding = encoder(
        docs=[query],
        normalize_embeddings=True
    )

    # Search Chroma for the most similar chunks
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    return results

In [14]:
test_questions = [
    "What are the main symptoms of insomnia?",
    "What factors can increase the risk of developing depression?",
    "What are the diagnostic features of autism spectrum disorder?",
    "What are the common symptoms of attention-deficit hyperactivity disorder?",
    "What are the main causes or risk factors for stroke?",
    "What are the common symptoms of epilepsy?",
    "What effects can sleep deprivation have on cognitive performance?",
    "What is rhythmic movement disorder and how does it present?",
    "What are the main challenges in managing neurological disorders?",
    "What screening methods are used to identify developmental problems in children?"
]


for i, question in enumerate(test_questions, start=1):

    results = retrieve_chunks(
        question,
        top_k=5
    )

    print("\n" + "=" * 100)
    print(f"Question {i}: {question}")
    print("=" * 100)

    for j in range(len(results["documents"][0])):

        metadata = results["metadatas"][0][j]

        print(f"\n--- Result {j + 1} ---")
        print(f"Distance: {results['distances'][0][j]:.4f}")
        print(f"Book: {metadata['book']}")
        print(
            f"Pages: "
            f"{metadata['page_start']}-{metadata['page_end']}"
        )
        print(f"Text: {results['documents'][0][j][:700]}")


Question 1: What are the main symptoms of insomnia?

--- Result 1 ---
Distance: 0.5050
Book: Sleep Disorders and Sleep Deprivation (National Academies)
Pages: 2254-2254
Text: INSOMNIA Manifestations and Prevalence Insomnia is the most commonly reported sleep problem (Ohayon, 2002). It is a highly prevalent disorder that often goes unrecognized and untreated despite its adverse impact on health and quality of life (Benca, 2005a) (see also Chapter 4). Insomnia is defined by having difficulty falling asleep, maintaining sleep, or by short sleep duration, despite adequate opportunity for a full night’s sleep. Other insomnia symptoms include daytime consequences, such as tiredness, lack of energy, difficulty concentrating, and/or irritability (Simon and VonKorff, 1997). The diagnostic criteria for primary insomnia include: • Difficulty initiating or maintaining sleep 

--- Result 2 ---
Distance: 0.5840
Book: Fundamentals of Psychological Disorders (mental use)
Pages: 999-999
Text: Insomnia

In [ ]:
test_questions_2 = [
    "What are the main diagnostic criteria for generalized anxiety disorder?",
    "What are the core symptoms of major depressive disorder?",
    "What are the main characteristics of intellectual developmental disorder?",
    "What are the early warning signs of developmental delay in children?",
    "What are the main causes and risk factors for dementia?",
    "What are the common symptoms and clinical features of migraine?",
    "How does chronic sleep deprivation affect memory and attention?",
    "What are the main treatments or management approaches for epilepsy?",
    "How is autism spectrum disorder distinguished from other neurodevelopmental disorders?",
    "What are the typical clinical features of sleep apnea?"
]

for i, question in enumerate(test_questions_2, start=1):

    results = retrieve_chunks(
        question,
        top_k=5
    )

    print("\n" + "=" * 100)
    print(f"Question {i}: {question}")
    print("=" * 100)

    for j in range(len(results["documents"][0])):

        print(f"\n--- Result {j + 1} ---")
        print(
            f"Distance: "
            f"{results['distances'][0][j]:.4f}"
        )

        print(
            f"Book: "
            f"{results['metadatas'][0][j]['book']}"
        )

        print(
            f"Pages: "
            f"{results['metadatas'][0][j]['page_start']}-"
            f"{results['metadatas'][0][j]['page_end']}"
        )

        print(
            f"Text: "
            f"{results['documents'][0][j][:1000]}"
        )


Question 1: What are the main diagnostic criteria for generalized anxiety disorder?

--- Result 1 ---
Distance: 0.5466
Book: WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders)
Pages: 285-285
Text: Course features • Onset of generalized anxiety disorder may occur at any age. However, the typical age of onset is during the early to mid-30s. • Earlier onset of symptoms is associated with greater impairment of functioning and presence of co-occurring mental disorders. • Severity of generalized anxiety disorder symptoms often fluctuates between threshold and subthreshold forms of the disorder, and full remission of symptoms is uncommon. • Although the clinical features of generalized anxiety disorder generally remain consistent across the lifespan, the content of the individual’s worry may vary over time, and there are differences in worry content among different age groups.

--- Result 2 ---
Distance: 0.5574
Book: WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopme

### RAG Prompting

This section builds the prompt used by the RAG system to generate answers from the retrieved evidence. The model is instructed to answer using only the provided context, avoid unsupported information, and cite the source book and page range for each relevant claim.

In [ ]:
import ollama


def build_rag_prompt(question, results):
    """Build a grounded RAG prompt from retrieved chunks."""

    context_parts = []

    for i in range(len(results["documents"][0])):

        document = results["documents"][0][i]
        metadata = results["metadatas"][0][i]

        source = (
            f"[Source {i + 1}] "
            f"{metadata['book']} "
            f"(pages {metadata['page_start']}-"
            f"{metadata['page_end']})"
        )

        context_parts.append(
            f"{source}\n{document}"
        )

    context = "\n\n".join(context_parts)

    SYSTEM_PROMPT = """YRole

You are NeuroAtlas, an assistant that answers questions about mental, neurodevelopmental, neurological, and sleep disorders. Your answers are grounded strictly in the provided document collection (the retrieved context passed to you for each query) — you are not answering from general world knowledge, and you never invent facts not present in the retrieved sources.

You help people understand conditions, symptoms, and general information so they can have a more informed conversation with a real healthcare provider.

Answer format

Answer the question directly and clearly.

For questions asking about symptoms, signs, features, causes, risk factors,
criteria, types, treatments, or other multiple items:

- Use clear bullet points.
- Put each distinct important point in its own bullet.
- Explain each point in a complete sentence or two.
- Group related points when appropriate.
- Do not turn the answer into several long paragraphs.
- Do not repeat the same information.
- Include the important relevant information found across the retrieved sources.

For simple questions with one clear answer:
- Use a short paragraph of 2–4 sentences.

For questions asking for an explanation:
- Start with a short definition or direct answer.
- Then use bullet points for the important details.

Do not force a fixed number of sentences or bullets.
The answer should be as detailed as the retrieved evidence requires.

Keep paragraphs compact and do not add unnecessary blank lines but leave one blank line between separate bullet points for readability.

Avoid:

One-line answers when the source material supports more depth.
be informative and stay stuff structured . speak in paragraphs not one big paragraph use bullet points when u need 
Dumping every retrieved sentence verbatim in a wall of bullets with no framing.
Repeating "[Source N]" inline after every clause — cite naturally (see below).

Match the depth of the answer to the question. "What are the symptoms of X" deserves a fuller breakdown than "Can X occur in adults," which may genuinely only need a sentence or two — but even short answers should sound complete, not clipped.

Grounding and citations
Only state facts that are supported by the retrieved documents for this query. If the collection doesn't cover something, say so plainly instead of guessing.
Cite sources by number inline where a specific claim needs attribution (e.g. "...associated with abnormalities in sleep microarchitecture (Source 3)"), rather than tacking a generic "📚 Sources (5)" onto the end with no indication of which source said what.
If sources disagree or only partially cover the question, say that explicitly rather than flattening it into one confident answer.
Handling unclear or misspelled input

If a term is misspelled or ambiguous (e.g. "eplispsey"):

Make a best-effort guess at the intended term and answer it, noting the correction briefly ("Assuming you mean epilepsy...").
Only ask for clarification if there's a genuine ambiguity between two different plausible terms — don't dead-end on a typo you can reasonably resolve yourself.
Tone

Warm, clear, and precise. Use everyday language first, then the clinical term, not the reverse. Avoid sounding like a search result — write as if explaining this to someone who actually wants to understand it, including someone who may be asking because it affects them or someone they care about.

Sensitive-topic handling
Never provide a diagnosis for an individual. Frame information at the condition level, not "you have X."
When a question could relate to the person's own health or a loved one's, gently suggest that a licensed clinician is the right next step for anything beyond general understanding — without being repetitive about it on every single answer.
If a message suggests acute distress, self-harm risk, or crisis, do not answer the informational question in isolation — respond supportively and note that if they're going through something difficult right now, reaching out to a crisis line or emergency services is the right move.

User question:
{question}

Retrieved context:
{context}
Reminder: cite every factual claim you make using [Source N], matching the source labels above exactly. Do not write a bullet or sentence with a factual claim and no citation.


Answer:
"""

    return SYSTEM_PROMPT.format(question=question, context=context)

RAG Answer Generation

This cell retrieves the most relevant chunks for a user question, builds a grounded RAG prompt from the retrieved evidence, and sends the prompt to the local `llama3.2` model.

The generated answer and the retrieved evidence are both returned so the answer can be evaluated against its supporting sources.

In [82]:
def generate_rag_answer(question, top_k=5):
    """Retrieve evidence and generate a grounded answer."""

    # Retrieve the most relevant chunks from Chroma
    results = retrieve_chunks(
        question,
        top_k=top_k
    )

    # Build the prompt from the retrieved evidence
    prompt = build_rag_prompt(
        question,
        results
    )

    # Generate the answer using the local LLM
    response = ollama.chat(
        model="llama3.2:latest",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": 0,
            "num_ctx": 4096,  # raise from Ollama's 2048 default — this long system prompt + 5 chunks needs the room
        }
    )

    # Extract the generated answer
    answer = response["message"]["content"]

    # Return the answer and retrieved evidence
    return answer, results

In [ ]:
while True:
    ## question = "What are the main symptoms of ADHD?"
    question = input("\nEnter your question (q to exit): ")

    if question.lower() == "q":
        print("Exiting...")
        break

    answer, results = generate_rag_answer(
        question,
        top_k=5
    )

    print("=" * 80)
    print("Question:")
    print(question)

    print("\n" + "=" * 80)
    print("Generated Answer:")
    print(answer)

    print("\n" + "=" * 80)
    print("Retrieved Sources:")

    for i in range(len(results["documents"][0])):

        metadata = results["metadatas"][0][i]

        print(
            f"[Source {i + 1}] "
            f"{metadata['book']} | "
            f"Pages {metadata['page_start']}-"
            f"{metadata['page_end']}"
        )

In [89]:
# question = "Can RMD occur in adults?"
question="What are the symptoms of OCD?"

answer, results = generate_rag_answer(
    question,
    top_k=5
)

print("=" * 80)
print("Question:")
print(question)

print("\n" + "=" * 80)
print("Generated Answer:")
print(answer)

print("\n" + "=" * 80)
print("Retrieved Sources:")

for i in range(len(results["documents"][0])):

    metadata = results["metadatas"][0][i]

    print(
        f"[Source {i + 1}] "
        f"{metadata['book']} | "
        f"Pages {metadata['page_start']}-"
        f"{metadata['page_end']}"
    )

Question:
What are the symptoms of OCD?

Generated Answer:
The symptoms of Obsessive-Compulsive Disorder (OCD) include:

• Recurring, intrusive thoughts, images, or impulses (obsessions): These can be distressing and unwanted, and may involve themes such as contamination, symmetry, or harm to oneself or others.
• Repeated behaviors or mental acts aimed at reducing distress (compulsions): These can be physical, such as handwashing or checking, or mental, such as repeating certain words or phrases.
• Preoccupation with symmetry, order, or exactness: Individuals with OCD may experience a strong need for symmetry, order, or exactness, which can manifest in behaviors such as arranging objects in a specific way or following strict routines.
• Excessive cleaning or hygiene behaviors: OCD can involve excessive cleaning or hygiene behaviors, such as handwashing or showering multiple times a day.
• Checking behaviors: Individuals with OCD may engage in repetitive checking behaviors, such as chec

In [84]:
test_questions_4 = [
    "What are the essential features required to diagnose generalized anxiety disorder?",
    "What symptoms are typically present in a major depressive episode?",
    "How can autism spectrum disorder present in young children?",
    "What are the main features of attention-deficit hyperactivity disorder?",
    "What are the signs that a child may have a developmental delay?",
    "What factors are associated with an increased risk of dementia?",
    "What does a typical migraine attack involve?",
    "What treatment options are available for people with epilepsy?",
    "Can rhythmic movement disorder continue beyond childhood?",
    "What are the typical symptoms of obstructive sleep apnea?",
    "How does lack of sleep affect attention and reaction time?",
    "What distinguishes autism spectrum disorder from conditions involving neurological regression?",
    "What are the main causes of disorders of intellectual development?",
    "What are the different ways insomnia can affect a person's daytime functioning?",
    "What are the main risk factors associated with stroke?",
    "How is epilepsy different from a single provoked seizure?",
    "What developmental skills are monitored when assessing a child's development?",
    "What are the main types of headache disorders?",
    "How can sleep disorders affect cognitive performance?",
    "What factors can make epilepsy difficult to manage in some populations?"
]


for i, question in enumerate(test_questions_4, start=1):

    print("\n" + "=" * 100)
    print(f"Question {i}: {question}")
    print("=" * 100)

    answer, results = generate_rag_answer(
        question,
        top_k=5
    )

    print("\nGenerated Answer:")
    print(answer)

    print("\nRetrieved Sources:")

    for j in range(len(results["documents"][0])):

        metadata = results["metadatas"][0][j]

        print(
            f"[Source {j + 1}] "
            f"{metadata['book']} | "
            f"Pages {metadata['page_start']}-"
            f"{metadata['page_end']}"
        )


Question 1: What are the essential features required to diagnose generalized anxiety disorder?



Generated Answer:
To diagnose generalized anxiety disorder, the following essential features are required:

• Marked symptoms of anxiety are necessary for diagnosis, which can manifest in two ways:
  • General apprehensiveness that is not restricted to any particular environmental circumstance, known as "free-floating anxiety".
  • Excessive worry (apprehensive expectation) about negative events occurring in several different aspects of everyday life, such as work, finances, health, or family.

• Anxiety and general apprehensiveness or worry are accompanied by additional characteristic symptoms, which may include:
  • Restlessness or feeling on edge
  • Fatigue or feeling tired
  • Difficulty concentrating
  • Irritability
  • Muscle tension
  • Sleep disturbances

• The symptoms must result in significant distress or significant impairment in personal, family, social, educational, occupational, or other important areas of functioning. If functioning is maintained, it is only through 

In [62]:
results = retrieve_chunks("What are the symptoms of OCD obsessive-compulsive disorder?", top_k=5)
for m, d in zip(results["metadatas"][0], results["documents"][0]):
    print(m["book"], m["page_start"], m["page_end"])
    print(d[:300])
    print("---")

WHO ICD-11 CDDR (Mental, Behavioural and Neurodevelopmental Disorders) 322 323
• Onset or exacerbation of obsessive-compulsive disorder has been reported during the peripartum period. Obsessive-compulsive and related disorders | Obsessive-compulsive disorder Obsessive-compulsive and related disorders Boundaries with other disorders and conditions (differential diagnosis) Bound
---
Fundamentals of Psychological Disorders (mental use) 1011 1011
1. What are the biological implications regarding the etiology of OCD and related disorders? What brain structures have been linked to these disorders? 2. Discuss identified cognitive biases that are related to the development and maintenance of OCD and related disorders? 3. The behavioral model dis
---
Fundamentals of Psychological Disorders (mental use) 1012 1012
Treatment will start at those with the lowest amount of distress to ensure the patient has success with treatment, as well as preventing withdrawal of treatment. Within the hierarchy of

| Topic Accuracy / Context Bleed | 19/20 clean; 1/20 showed a known cross-topic contamination failure in Q8, where an aspirin/stroke-treatment sentence was incorrectly included in an epilepsy-treatment answer. The implemented topic-accuracy prompt rule reduced this issue but did not eliminate it reliably. This demonstrates that prompt-based mitigation with the local 3B model is probabilistic rather than guaranteed. The failure is documented as a known limitation, with retrieval-side filtering/tighter chunking identified as a potential future improvement. |

## 2.5 Vision Component

 Fine-tune YOLOv8-cls on the facial image dataset, evaluate it, then decide
how its prediction (label + confidence) gets fused into the RAG prompt context.

Importing datasets

In [18]:
import kagglehub


autism_path = kagglehub.dataset_download("meimeizhong/facial-dataset-of-autistic-children")

print("Path to dataset files:", autism_path)



Path to dataset files: C:\Users\mm\.cache\kagglehub\datasets\meimeizhong\facial-dataset-of-autistic-children\versions\1


In [19]:
import kagglehub

down_path = kagglehub.dataset_download(
    "mamunhasan2cs/down-syndrome-dataset"
)

print("Dataset downloaded to:")
print(down_path)



Dataset downloaded to:
C:\Users\mm\.cache\kagglehub\datasets\mamunhasan2cs\down-syndrome-dataset\versions\1


In [20]:
import kagglehub

# Download latest version
depression_path = kagglehub.dataset_download("khairunneesa/depression-dataset-on-facial-ecpression-images")

print("Path to dataset files:", depression_path)

Path to dataset files: C:\Users\mm\.cache\kagglehub\datasets\khairunneesa\depression-dataset-on-facial-ecpression-images\versions\1


Importing libraries

In [21]:
from ultralytics import YOLO
import os
import shutil
import random
from pathlib import Path
from IPython.display import display, Image
from IPython import display
display.clear_output()
!yolo mode=checks

# Fix the random seed so our train/val split is reproducible.
random.seed(0)

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\.venv312\Scripts\yolo.exe\__main__.py", line 7, in <module>
  File "C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\.venv312\Lib\site-packages\ultralytics\cfg\__init__.py", line 1083, in entrypoint
    raise ValueError(f"Invalid 'mode={mode}'. Valid modes are {list(MODES)}.\n{CLI_HELP_MSG}")
ValueError: Invalid 'mode=checks'. Valid modes are ['train', 'val', 'predict', 'export', 'track', 'benchmark'].

    Arguments received: ['yolo', 'mode=checks']. Ultralytics 'yolo' commands use the following syntax:

        yolo TASK MODE ARGS

        Where   TASK (optional) is one of ['detect', 'segment', 'semantic', 'depth', 'classify', 'pose', 'obb']
                MODE (required) is one of ['train', 'val', 

In [22]:
from pathlib import Path

DATASET_PATHS = {
    "down_syndrome": down_path,
    "autism": autism_path,
    "depression": depression_path,
}


# ACTIVE_DATASET = "down_syndrome" 

# raw_dataset_dir = Path(
#     DATASET_PATHS[ACTIVE_DATASET]
# )

# print(f"Using dataset: {ACTIVE_DATASET}")
# print(f"Raw path: {raw_dataset_dir}")

folder inspection

In [ ]:
# # Walk the downloaded folder and print every subfolder with its image count,
# # so we can see exactly how kagglehub laid the classes out before we touch anything.

# def inspect_image_folder(root_dir, max_depth=3):

#     root_dir = Path(root_dir)

#     # Walk every subdirectory under the root.
#     for dirpath, dirnames, filenames in os.walk(root_dir):

#         # Only count files that look like images.
#         image_files = [f for f in filenames if f.lower().endswith((".jpg", ".jpeg", ".png"))]

#         # Skip printing empty non-leaf folders to keep the output readable.
#         if image_files:
#             rel_path = Path(dirpath).relative_to(root_dir)
#             print(f"{rel_path}  ->  {len(image_files)} images")


# inspect_image_folder(raw_dataset_dir)

1. Build YOLO-cls Datasets

Creates a separate YOLO classification dataset for each of the three datasets.Preserves existing train/validation splits or creates them when needed. Stores all three datasets separately so the router can select the appropriate model during inference.


In [24]:
def build_yolo_cls_dataset(raw_dir, output_dir, val_ratio=0.2):
    """Build a YOLO-cls dataset (train/<class>, val/<class>) from either:
    (a) a raw_dir that ALREADY has train/val/(test) subfolders per class
        -> just copy the existing splits as-is, preserving them.
    (b) a raw_dir with flat class folders and no split
        -> create a fresh random train/val split (val_ratio).
    """

    raw_dir = Path(raw_dir)
    output_dir = Path(output_dir)

    # Start clean in case this cell is re-run.
    if output_dir.exists():
        shutil.rmtree(output_dir)

    # Check whether the dataset already has train/ and val/ top-level folders
    # (Autism and Depression both do; Down Syndrome does not).
    has_existing_split = (raw_dir / "train").is_dir() and (raw_dir / "val").is_dir()

    # Depression's dataset nests everything one level deeper, under "data/".
    # Detect that and step into it before checking for train/val.
    if not has_existing_split and (raw_dir / "data" / "train").is_dir():
        raw_dir = raw_dir / "data"
        has_existing_split = True

    if has_existing_split:
        print("Existing train/val split detected — copying as-is.")

        # Only copy the splits YOLO-cls needs (train, val). Test is left alone —
        # use it later for a final held-out evaluation, not during training.
        for split in ["train", "val"]:
            split_dir = raw_dir / split

            for class_dir in split_dir.iterdir():
                if not class_dir.is_dir():
                    continue

                image_files = [f for f in class_dir.iterdir()
                               if f.suffix.lower() in (".jpg", ".jpeg", ".png")]

                dest_dir = output_dir / split / class_dir.name
                dest_dir.mkdir(parents=True, exist_ok=True)

                for f in image_files:
                    shutil.copy(f, dest_dir / f.name)

                print(f"  {split}/{class_dir.name}: {len(image_files)} images")

    else:
        print("No existing split detected — creating a random train/val split.")

        # Find every leaf folder that directly contains image files — each one is a class.
        class_folders = []
        for dirpath, dirnames, filenames in os.walk(raw_dir):
            image_files = [f for f in filenames if f.lower().endswith((".jpg", ".jpeg", ".png"))]
            if image_files:
                class_folders.append((Path(dirpath), image_files))

        print(f"Found {len(class_folders)} class folders.")

        for class_dir, image_files in class_folders:
            class_name = class_dir.name
            random.shuffle(image_files)

            val_count = max(1, int(len(image_files) * val_ratio))
            val_files = image_files[:val_count]
            train_files = image_files[val_count:]

            train_dest = output_dir / "train" / class_name
            val_dest = output_dir / "val" / class_name
            train_dest.mkdir(parents=True, exist_ok=True)
            val_dest.mkdir(parents=True, exist_ok=True)

            for f in train_files:
                shutil.copy(class_dir / f, train_dest / f)
            for f in val_files:
                shutil.copy(class_dir / f, val_dest / f)

            print(f"  {class_name}: {len(train_files)} train, {len(val_files)} val")

    return output_dir 

2. Build YOLO-Classification Datasets

Build a separate YOLO-classification dataset for each of the three datasets.Each prepared dataset is stored in `yolo_dirs`, keyed by its dataset name. This keeps all three YOLO-ready datasets available on disk so that the corresponding trained model can be used later by the inference router.


In [25]:
yolo_dirs = {}

for dataset_name, raw_path in DATASET_PATHS.items():

    print(f"\n{'='*60}")
    print(f"Building YOLO-cls layout for: {dataset_name}")
    print(f"{'='*60}")

    # reates the YOLO-ready version of each dataset and stores its path in yolo_dirs.
    #This lets the training loop access all three prepared datasets separately.
    yolo_dirs[dataset_name] = build_yolo_cls_dataset(
        raw_dir=Path(raw_path),
        output_dir=Path("data") / f"yolo_cls_{dataset_name}",
        val_ratio=0.2,
    )

print("\nAll YOLO-cls datasets ready:")
for name, path in yolo_dirs.items():
    print(f"  {name}: {path}")


Building YOLO-cls layout for: down_syndrome
No existing split detected — creating a random train/val split.
Found 2 class folders.
  downSyndrome: 1200 train, 300 val
  healthy: 1200 train, 299 val

Building YOLO-cls layout for: autism
Existing train/val split detected — copying as-is.
  train/Autistic: 864 images
  train/Non_Autistic: 873 images
  val/Autistic: 289 images
  val/Non_Autistic: 291 images

Building YOLO-cls layout for: depression
No existing split detected — creating a random train/val split.
Found 21 class folders.
  Angry: 48 train, 12 val
  Disgust: 39 train, 9 val
  Fear: 39 train, 9 val
  Happy: 40 train, 10 val
  Neutral: 58 train, 14 val
  Sad: 40 train, 10 val
  Surprize: 48 train, 12 val
  Angry: 461 train, 115 val
  Disgust: 252 train, 63 val
  Fear: 400 train, 100 val
  Happy: 442 train, 110 val
  Neutral: 596 train, 148 val
  Sad: 375 train, 93 val
  Surprize: 449 train, 112 val
  Angry: 48 train, 12 val
  Disgust: 39 train, 9 val
  Fear: 39 train, 9 val
  H

3. Train All YOLO Models

Trains a separate YOLO classification model for each dataset and saves the best checkpoint. The models remain available for the router to select during inference.


In [28]:
trained_models = {}

for dataset_name, yolo_dir in yolo_dirs.items():

    print(f"\n{'='*60}")
    print(f"Training on: {dataset_name}")
    print(f"{'='*60}")

    model = YOLO("yolov8n-cls.pt")
    results = model.train(
        data=str(yolo_dir),
        epochs=50,
        imgsz=224,
        batch=16,
        patience=15,
        project="runs",
        name=f"{dataset_name}_cls",
    )

    best = YOLO(results.save_dir / "weights" / "best.pt")
    trained_models[dataset_name] = best

    print(f"{dataset_name} done. Weights: {results.save_dir / 'weights' / 'best.pt'}")

print("\nAll models trained:", list(trained_models.keys()))


Training on: down_syndrome
New https://pypi.org/project/ultralytics/8.4.148 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.146  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data\yolo_cls_down_syndrome, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=

In [57]:
for dataset_name, model in trained_models.items():
    print(f"{dataset_name}:")
    print(model.ckpt_path)

down_syndrome:
C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\runs\classify\runs\down_syndrome_cls-6\weights\best.pt
autism:
C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\runs\classify\runs\autism_cls-5\weights\best.pt
depression:
C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\runs\classify\runs\depression_cls-5\weights\best.pt


In [60]:
from ultralytics import YOLO

for name, path in models.items():
    print(f"\nLoading {name}...")

    model = YOLO(str(path))

    print(f"✓ {name} loaded successfully")
    print("Classes:", model.names)


Loading down_syndrome...
✓ down_syndrome loaded successfully
Classes: {0: 'downSyndrome', 1: 'healthy'}

Loading autism...
✓ autism loaded successfully
Classes: {0: 'Autistic', 1: 'Non_Autistic'}

Loading depression...
✓ depression loaded successfully
Classes: {0: 'Angry', 1: 'Disgust', 2: 'Fear', 3: 'Happy', 4: 'Neutral', 5: 'Sad', 6: 'Surprize'}


4. Evaluate All YOLO Models

Validates each trained model and prints its Top-1 accuracy. This allows the performance of the three classifiers to be compared side by side.


In [29]:
for dataset_name, model in trained_models.items():
    metrics = model.val(verbose=False) #validate this trained YOLO model
    print(
        f"{dataset_name}: "
        f"top1 = {metrics.top1:.3f}, "
        f"top5 = {metrics.top5:.3f}"
    )

Ultralytics 8.4.146  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\data\yolo_cls_down_syndrome\train... found 2400 images in 2 classes  
val: C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\data\yolo_cls_down_syndrome\val... found 599 images in 2 classes  
test: None...
WARNING val: Slow image access detected (ping: 0.61.1 ms, read: 4.92.2 MB/s, size: 9.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\data\yolo_cls_down_syndrome\val... 599 images, 0 corrupt: 100% ━━━━━━━━━━━━ 599/599

5. Question-Based Dataset Router

This cell uses keywords in the user's question to select the relevant trained classifier. If no supported condition is detected, it returns `None` and skips the image classifier.


In [ ]:
# Simple keyword router. Given the user's question, decide which of the
# 3 trained classifiers is relevant. This runs BEFORE we touch the image.
# exmaple -> "Could this photo show signs of Down syndrome?" ->route_dataset()->"down_syndrome" ->Down Syndrome YOLO model


ROUTING_KEYWORDS = {
    "down_syndrome": [
        "down syndrome",
        "down's syndrome",
        "trisomy 21",
    ],
    "autism": [
        "autism",
        "autistic",
        "asd",
    ],
    "depression": [
        "depression",
        "depressed",
        "mood",
        "sad",
        "emotion",
    ],
}


def route_dataset(question: str):
    """Return the dataset key most relevant to the question, or None if
    nothing matches (caller should ask the user or skip CV).

    Counts keyword hits per category instead of returning on the first
    match found. This matters because dict iteration order previously
    decided the outcome for any question that happened to contain
    keywords from more than one category — e.g. "is my child's sad mood
    related to autism" contains both "sad"/"mood" (depression) and
    "autism" (autism), and the old version always picked whichever
    category was defined first in ROUTING_KEYWORDS regardless of which
    one the question was actually about.

    Rule: the category with the most keyword hits wins. If two or more
    categories are tied for the most hits, the match is genuinely
    ambiguous and we return None rather than silently guessing — the
    caller can then skip image classification or ask the user to
    clarify, instead of confidently routing to the wrong classifier.
    """

    question_lower = question.lower()

    hit_counts = {}

    for dataset_name, keywords in ROUTING_KEYWORDS.items():
        hits = sum(1 for keyword in keywords if keyword in question_lower)
        if hits > 0:
            hit_counts[dataset_name] = hits

    if not hit_counts:
        return None

    max_hits = max(hit_counts.values())
    top_matches = [name for name, hits in hit_counts.items() if hits == max_hits]

    if len(top_matches) > 1:
        # Genuine ambiguity between categories — don't guess.
        return None

    return top_matches[0]


# Quick tests.
print(route_dataset("Could this photo show signs of Down syndrome?"))   # -> down_syndrome
print(route_dataset("Does this face look autistic?"))                    # -> autism
print(route_dataset("Is this person showing signs of depression?"))      # -> depression
print(route_dataset("What causes insomnia?"))                            # -> None (no image relevant)

down_syndrome
autism
depression
None


6. Route Image to the Correct Classifier

This cell uses the user's question to select the appropriate trained YOLO model, then classifies the uploaded image. It returns the dataset used, predicted class, and confidence score; if the question does not match any supported dataset, no image classification is performed.


In [31]:
def classify_image(image_path, question, models=trained_models):
    """Route to the correct model based on the question, run inference,
    return (dataset_used, predicted_class, confidence) or None if no
    image-relevant dataset matched the question."""

    dataset_key = route_dataset(question)

    if dataset_key is None:
        return None  # question isn't about any of our 3 image conditions

    model = models[dataset_key]
    result = model(image_path)[0] #runs the selected model on the uploaded image.

    top_idx = result.probs.top1 #Gets the index of the class with the highest probability.
    predicted_class = result.names[top_idx]
    confidence = float(result.probs.top1conf)

    return dataset_key, predicted_class, confidence


# Smoke test.
sample_path = next(Path("data/yolo_cls_autism/val").glob("*/*.*"))
print(classify_image(sample_path, "Does this child show signs of autism?"))


image 1/1 c:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\data\yolo_cls_autism\val\Autistic\0007.jpg: 224x224 Autistic 1.00, Non_Autistic 0.00, 5.4ms
Speed: 29.3ms preprocess, 5.4ms inference, 1.0ms postprocess per image at shape (1, 3, 224, 224)
('autism', 'Autistic', 0.9999849796295166)


7. Multimodal RAG Prompt

Combines the retrieved text sources with the relevant YOLO image classification result. The image finding is added as a labeled source so the LLM can use it alongside the RAG evidence while treating it as a non-diagnostic visual result.


**Decision:** The classifier's output (predicted class + confidence) is formatted as an extra labeled source — **[Image Analysis]** — and appended to the retrieved text context before it reaches the LLM. This keeps the same grounding and citation pattern as the text sources, while explicitly instructing the LLM to treat the visual result as an unverified, non-diagnostic classification rather than a confirmed clinical diagnosis.


In [32]:
def build_multimodal_rag_prompt(question, results, image_path=None, models=trained_models):
    """Build a grounded RAG prompt. If image_path is given, route it through
    the right classifier (based on the question) and add it as a cited source."""

    context_parts = []

    for i in range(len(results["documents"][0])):
        document = results["documents"][0][i]
        metadata = results["metadatas"][0][i]
        source = (
            f"[Source {i + 1}] "
            f"{metadata['book']} "
            f"(pages {metadata['page_start']}-{metadata['page_end']})"
        )
        context_parts.append(f"{source}\n{document}")

    image_section = ""
    if image_path is not None:
        classification = classify_image(image_path, question, models)

        if classification is not None:
            dataset_key, predicted_class, confidence = classification
            image_source_number = len(results["documents"][0]) + 1
            image_section = (
                f"\n\n[Source {image_source_number}] Image Analysis "
                f"({dataset_key.replace('_', ' ')} classifier, YOLO)\n"
                f"Predicted class: {predicted_class} (confidence: {confidence:.1%}). "
            )
        # If classification is None, the question wasn't about any of our
        # 3 supported conditions — we silently skip the image, no image_section added.

    context = "\n\n".join(context_parts) + image_section
    prompt = f"""
You are a scientific mental-health RAG assistant.

Answer the user's question using the retrieved sources and the image
classification result.

CLASSIFICATION RULES:

1. DOWN SYNDROME
- downSyndrome -> "The image is predicted as Down syndrome (X% confidence)."
- healthy -> "The image is predicted as healthy (X% confidence). Therefore,
  based on the classifier result, the image does not indicate Down syndrome."
- Then explain the relevant Down syndrome information from the sources.

2. AUTISM
- Autistic -> "The image is predicted as Autistic (X% confidence)."
- Non_Autistic -> "The image is predicted as Non_Autistic (X% confidence).
  Therefore, based on the classifier result, the image does not indicate autism."
- Then explain the relevant autism information from the sources.

3. DEPRESSION DATASET

This classifier is used to identify facial emotional patterns relevant to depression.

- State the detected emotion and confidence exactly as provided.
- Use the retrieved sources to determine whether the detected emotion is related to depression.
- If the sources support a relationship, explain how the emotion is relevant to depression.
- If the sources do not support a relationship, state that the detected emotion is not shown to be related to depression by the retrieved evidence, then explain the emotion itself using the available sources.
- Then explain the relevant depression information from the sources.
- Do not invent information that is not supported by the sources.
- If the emotion is irrelevant to deppression, explain the emotion from your knowledge

GENERAL RULES:
- Start directly with the classification result.
- State the classification result only once.
- Never change the predicted class/emotion or confidence.
- Never call confidence accuracy.
- Never add generic medical disclaimers.
- Never say the classifier accurately identified the person.
- Never repeat or reinterpret the classification result later.
- Use only information supported by the retrieved sources.
- Cite factual claims as [Source N].
- Only use source numbers that exist in the retrieved context.
- Explain the information clearly and in enough detail to answer the
  question. Do not make the answer unnecessarily short.
- Use connected explanations rather than isolated one-line facts.
- Do not use "Answer:".
- Do not reproduce "User question:" or "Retrieved context:".
- Never use the phrases:
  "however", "it is essential to note", "it's essential to note",
  "it is important to note", "it's important to note",
  "does not necessarily determine", "does not necessarily indicate",
  "does not necessarily mean", "should not be used as the sole basis
  for diagnosis", "should not be relied upon",
  "this is not a definitive diagnostic tool", "automated screening tool".

OUTPUT STRUCTURE:

Image classification:
[2-3 sentences stating the classification result and a brief,
relevant interpretation.]

Interpretation:
[2-4 sentences explaining the result in relation to the user's
question. Do not repeat the classification result.]

Relevant information:
- [Detailed factual point supported by a source.]
- [Detailed factual point supported by a source.]
- [Detailed factual point supported by a source.]

The following are INPUTS, not output sections:

User question:
{question}

Retrieved context:
{context}

Generate ONLY the final answer.
"""
    return prompt.format(question=question, context=context)

<!-- Example: a user asks a question AND uploads a facial image.

The router (inside classify_image) automatically decides which of the 3 fine-tuned models to use, based on keywords in the question itself —
the user doesn't have to specify a condition manually.

question = "Could this image be related to Down syndrome, and what are the typical symptoms?"
uploaded_image_path = sample_image  # stand-in for a real user upload

# 1. Retrieve text context as usual (your existing retrieval function).
results = retrieve(question, top_k=5)   # <- your Section 2.4 retrieval function

# 2. Build the fused multimodal prompt. classify_image() is called internally —
#    it routes the question to the right model (down_syndrome / autism / depression),
#    runs inference, and folds the result into the context as a cited source.
#    If the question isn't about any of the 3 supported conditions, the image
#    is silently skipped and the prompt falls back to text-only context.
prompt = build_multimodal_rag_prompt(question, results, image_path=uploaded_image_path)

# 3. Call Ollama as usual.
response = ollama.generate(model="your-model-name", prompt=prompt)
print(response["response"]) -->

In [44]:
#Evaluate
import random
from collections import defaultdict

def evaluate_classifier(dataset_name, model, yolo_dir, n_per_class=30):
    """Run the classifier on n_per_class images per class and report
    per-class accuracy + a confusion matrix. No LLM involved."""

    val_dir = Path(yolo_dir) / "val"
    class_dirs = sorted([d for d in val_dir.iterdir() if d.is_dir()])
    class_names = [d.name for d in class_dirs]

    confusion = defaultdict(lambda: defaultdict(int))
    per_class_correct = defaultdict(int)
    per_class_total = defaultdict(int)

    for class_dir in class_dirs:
        images = list(class_dir.glob("*.*"))
        random.shuffle(images)
        sample = images[:n_per_class]

        for img_path in sample:
            result = model(img_path, verbose=False)[0]
            predicted = result.names[result.probs.top1]

            confusion[class_dir.name][predicted] += 1
            per_class_total[class_dir.name] += 1
            if predicted == class_dir.name:
                per_class_correct[class_dir.name] += 1

    print(f"\n=== {dataset_name} ===")
    total_correct = sum(per_class_correct.values())
    total = sum(per_class_total.values())
    print(f"Overall accuracy: {total_correct}/{total} = {total_correct/total:.1%}\n")

    print("Per-class accuracy:")
    for name in class_names:
        c, t = per_class_correct[name], per_class_total[name]
        print(f"  {name:15s}: {c}/{t} = {c/t:.1%}" if t else f"  {name}: no samples")

    print("\nConfusion matrix (rows=true, cols=predicted):")
    header = "true\\pred".ljust(15) + "".join(n[:10].ljust(11) for n in class_names)
    print(header)
    for true_name in class_names:
        row = true_name.ljust(15)
        for pred_name in class_names:
            row += str(confusion[true_name][pred_name]).ljust(11)
        print(row)

    return confusion


for dataset_name, model in trained_models.items():
    evaluate_classifier(dataset_name, model, yolo_dirs[dataset_name], n_per_class=30)


=== down_syndrome ===
Overall accuracy: 58/60 = 96.7%

Per-class accuracy:
  downSyndrome   : 29/30 = 96.7%
  healthy        : 29/30 = 96.7%

Confusion matrix (rows=true, cols=predicted):
true\pred      downSyndro healthy    
downSyndrome   29         1          
healthy        1          29         

=== autism ===
Overall accuracy: 50/60 = 83.3%

Per-class accuracy:
  Autistic       : 23/30 = 76.7%
  Non_Autistic   : 27/30 = 90.0%

Confusion matrix (rows=true, cols=predicted):
true\pred      Autistic   Non_Autist 
Autistic       23         7          
Non_Autistic   3          27         

=== depression ===
Overall accuracy: 137/210 = 65.2%

Per-class accuracy:
  Angry          : 17/30 = 56.7%
  Disgust        : 23/30 = 76.7%
  Fear           : 11/30 = 36.7%
  Happy          : 24/30 = 80.0%
  Neutral        : 22/30 = 73.3%
  Sad            : 14/30 = 46.7%
  Surprize       : 26/30 = 86.7%

Confusion matrix (rows=true, cols=predicted):
true\pred      Angry      Disgust    Fear       

In [38]:
from pathlib import Path
from ultralytics import YOLO

BASE_DIR = Path(
    r"C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project"
)

MODEL_DIR = BASE_DIR / "backend" / "models"
DATA_DIR = BASE_DIR / "notebooks" / "data"

models = {
    "autism": MODEL_DIR / "autsim_best.pt",
    "depression": MODEL_DIR / "depression_best.pt",
    "down_syndrome": MODEL_DIR / "down_syndrome_best.pt",
}

datasets = {
    "autism": DATA_DIR / "yolo_cls_autism",
    "depression": DATA_DIR / "yolo_cls_depression",
    "down_syndrome": DATA_DIR / "yolo_cls_down_syndrome",
}

for name in models:

    print("\n" + "=" * 60)
    print(f"Evaluating: {name}")
    print("=" * 60)

    model = YOLO(str(models[name]))

    results = model.val(
        data=str(datasets[name]),
        split="val",
        workers=0
    )

    print(f"Top-1 Accuracy: {results.top1 * 100:.2f}%")


Evaluating: autism
Ultralytics 8.4.146  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\data\yolo_cls_autism\train... found 1737 images in 2 classes  
val: C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\data\yolo_cls_autism\val... found 580 images in 2 classes  
test: None...
WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 19.415.1 MB/s, size: 21.8 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\data\yolo_cls_autism\val... 580 images, 0 corrupt: 100% ━━━━━━━━━━━━ 580/5

               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 37/37 5.7it/s 6.5s<0.2s
                   all      0.834          1
Speed: 0.2ms preprocess, 2.7ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\runs\classify\val-15
Top-1 Accuracy: 83.45%

Evaluating: depression
Ultralytics 8.4.146  Python-3.12.10 torch-2.6.0+cu124 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,443,847 parameters, 0 gradients, 3.3 GFLOPs
train: C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\data\yolo_cls_depression\train... found 3049 images in 7 classes  
val: C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\data\yolo_cls_depression\val... found 852 images in 7 classes  
test: None...
WARNING val: Slow image access 

In [34]:
for d in (Path(yolo_dirs["depression"]) / "train").iterdir():
    print(d.name, len(list(d.glob("*.*"))))

Angry 474
Disgust 262
Fear 405
Happy 453
Neutral 608
Sad 385
Surprize 462


In [35]:
#test 1
# Sanity-check the keyword router in isolation before touching any images.

router_test_cases = [
    ("Could this photo show signs of Down syndrome?", "down_syndrome"),
    ("Does this child's face look autistic?", "autism"),
    ("Is this person showing signs of depression?", "depression"),
    ("What is trisomy 21?", "down_syndrome"),
    ("What causes insomnia?", None),
    ("Explain the symptoms of generalized anxiety disorder.", None),
]

for question, expected in router_test_cases:
    result = route_dataset(question)
    status = "OK" if result == expected else "MISMATCH"
    print(f"[{status}] '{question}' -> {result} (expected {expected})")

[OK] 'Could this photo show signs of Down syndrome?' -> down_syndrome (expected down_syndrome)
[OK] 'Does this child's face look autistic?' -> autism (expected autism)
[OK] 'Is this person showing signs of depression?' -> depression (expected depression)
[OK] 'What is trisomy 21?' -> down_syndrome (expected down_syndrome)
[OK] 'What causes insomnia?' -> None (expected None)
[OK] 'Explain the symptoms of generalized anxiety disorder.' -> None (expected None)


<!-- Example: a user asks a question AND uploads a facial image.

The router (inside classify_image) automatically decides which of the 3 fine-tuned models to use, based on keywords in the question itself —
the user doesn't have to specify a condition manually.

question = "Could this image be related to Down syndrome, and what are the typical symptoms?"
uploaded_image_path = sample_image  # stand-in for a real user upload

# 1. Retrieve text context as usual (your existing retrieval function).
results = retrieve(question, top_k=5)   # <- your Section 2.4 retrieval function

# 2. Build the fused multimodal prompt. classify_image() is called internally —
#    it routes the question to the right model (down_syndrome / autism / depression),
#    runs inference, and folds the result into the context as a cited source.
#    If the question isn't about any of the 3 supported conditions, the image
#    is silently skipped and the prompt falls back to text-only context.
prompt = build_multimodal_rag_prompt(question, results, image_path=uploaded_image_path)

# 3. Call Ollama as usual.
response = ollama.generate(model="your-model-name", prompt=prompt)
print(response["response"]) -->

In [41]:
# etst 2
# Pull a few images the model has never trained/validated on and check
# predicted vs. true label, per dataset.

def test_classifier_accuracy(dataset_name, model, yolo_dir, n_samples=10):
    """Run the model on n_samples random val images and report accuracy."""

    val_dir = Path(yolo_dir) / "val"
    class_dirs = [d for d in val_dir.iterdir() if d.is_dir()]

    correct = 0
    total = 0
    samples_checked = []

    for class_dir in class_dirs:
        images = list(class_dir.glob("*.*"))
        random.shuffle(images)

        for img_path in images[:n_samples // len(class_dirs) + 1]:
            result = model(img_path, verbose=False)[0]
            predicted = result.names[result.probs.top1]
            confidence = float(result.probs.top1conf)

            is_correct = predicted == class_dir.name
            correct += is_correct
            total += 1

            samples_checked.append((img_path.name, class_dir.name, predicted, confidence, is_correct))

    print(f"\n{dataset_name}: {correct}/{total} correct on sample check")
    for name, true_label, pred_label, conf, ok in samples_checked:
        mark = "✓" if ok else "✗"
        print(f"  {mark} {name[:30]:30s} true={true_label:15s} pred={pred_label:15s} conf={conf:.1%}")


for dataset_name, model in trained_models.items():
    test_classifier_accuracy(dataset_name, model, yolo_dirs[dataset_name])


down_syndrome: 12/12 correct on sample check
  ✓ down_97.jpg                    true=downSyndrome    pred=downSyndrome    conf=98.0%
  ✓ down_147.jpg                   true=downSyndrome    pred=downSyndrome    conf=99.6%
  ✓ down_323.jpg                   true=downSyndrome    pred=downSyndrome    conf=100.0%
  ✓ down_500.jpg                   true=downSyndrome    pred=downSyndrome    conf=100.0%
  ✓ down_56.jpeg                   true=downSyndrome    pred=downSyndrome    conf=99.8%
  ✓ down_1308.jpg                  true=downSyndrome    pred=downSyndrome    conf=100.0%
  ✓ healty_378.jpg                 true=healthy         pred=healthy         conf=100.0%
  ✓ healty_497.jpg                 true=healthy         pred=healthy         conf=100.0%
  ✓ healty_768.jpg                 true=healthy         pred=healthy         conf=99.6%
  ✓ healty_37.jpg                  true=healthy         pred=healthy         conf=96.3%
  ✓ healty_290.jpg                 true=healthy         pred=healthy 

In [42]:
#test 4
# Make sure a question with no image-relevant keywords does NOT trigger
# any classifier, even with an image attached — this should silently
# fall back to text-only context.

unrelated_question = "What are the main symptoms of insomnia?"
some_image = next((Path(yolo_dirs["down_syndrome"]) / "val").glob("*/*.*"))

routing_result = classify_image(some_image, unrelated_question)
print("Routing result (should be None):", routing_result)

prompt = build_multimodal_rag_prompt(unrelated_question, retrieve_chunks(unrelated_question, top_k=5), image_path=some_image)
print("\n[Image Analysis]" in prompt)  # should print False — no image section added

Routing result (should be None): None
False


In [ ]:
#test 3
import random

# Make the random image selection reproducible
random.seed(42)

# Run the whole question + image -> classify -> RAG prompt -> Ollama flow
# for the selected test cases using real validation images.

test_cases = [
    ("down_syndrome", "Could this image be related to Down syndrome, and what are the typical symptoms?"),
    ("down_syndrome", "What are the common characteristics associated with Down syndrome?"),
    ("down_syndrome", "What medical and developmental features are commonly associated with Down syndrome?"),

    ("autism", "Does this image show signs consistent with autism, and what are common symptoms?"),
    ("autism", "What are the common characteristics and symptoms associated with autism?"),
    ("autism", "What behavioral and developmental features are commonly associated with autism?"),

    ("depression", "Could this image show signs of depression, and what are the typical symptoms?"),
    ("depression", "What are the common symptoms and characteristics of depression?"),
    ("depression", "What behavioral and emotional symptoms are commonly associated with depression?"),
    ("depression", "What are some common signs associated with depression?")
]


for i, (dataset_name, question) in enumerate(test_cases, start=1):

    val_dir = Path(yolo_dirs[dataset_name]) / "val"

    # Get all class folders
    class_dirs = [
        d for d in val_dir.iterdir()
        if d.is_dir()
    ]

    # Randomly choose one class
    sample_class_dir = random.choice(class_dirs)

    # Randomly choose one image from that class
    images = list(sample_class_dir.glob("*.*"))
    sample_img = random.choice(images)

    print(f"\n{'=' * 80}")
    print(f"TEST CASE {i}/{len(test_cases)}")
    print(f"Dataset: {dataset_name}")
    print(f"Question: {question}")
    print(f"Image: {sample_img.name}")
    print(f"True class: {sample_class_dir.name}")
    print(f"{'=' * 80}")

    # --------------------------------------------------
    # 1. Classify the image
    # --------------------------------------------------
    routing_result = classify_image(
        sample_img,
        question
    )

    print("\nRouting result:")
    print(routing_result)

    # --------------------------------------------------
    # 2. Retrieve relevant RAG chunks
    # --------------------------------------------------
    results = retrieve_chunks(
        question,
        top_k=5
    )

    # --------------------------------------------------
    # 3. Build the multimodal RAG prompt
    # --------------------------------------------------
    prompt = build_multimodal_rag_prompt(
        question,
        results,
        image_path=sample_img
    )

    # --------------------------------------------------
    # 4. Generate the final answer with Ollama
    # --------------------------------------------------
    response = ollama.generate(
        model="llama3.2",
        prompt=prompt,
        options={"temperature": 0}
    )

    print("\nAnswer:")
    print(response["response"])


TEST CASE 1/10
Dataset: down_syndrome
Question: Could this image be related to Down syndrome, and what are the typical symptoms?
Image: down_1062.jpg
True class: downSyndrome

image 1/1 c:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\data\yolo_cls_down_syndrome\val\downSyndrome\down_1062.jpg: 224x224 downSyndrome 0.93, healthy 0.07, 5.8ms
Speed: 2.6ms preprocess, 5.8ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)

Routing result:
('down_syndrome', 'downSyndrome', 0.9272436499595642)

image 1/1 c:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\data\yolo_cls_down_syndrome\val\downSyndrome\down_1062.jpg: 224x224 downSyndrome 0.93, healthy 0.07, 5.3ms
Speed: 3.1ms preprocess, 5.3ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)

Answer:
Image classification:
The image is predicted as Down syndrome (92.7% confidence).

Interpretation:
Down sy

## 2.6 Evaluation Report

In [49]:
# Mix of text-only questions (Core Track) and image-relevant questions
# (Extended Track) to demonstrate both parts of your pipeline.

eval_questions = [
    {"question": "What are the main symptoms of insomnia?", "image_path": None},
    {"question": "What are the essential features required to diagnose generalized anxiety disorder?", "image_path": None},
    {"question": "What are the main causes of disorders of intellectual development?", "image_path": None},
    {"question": "What are the main features of attention-deficit hyperactivity disorder?", "image_path": None},
    {"question": "What are the typical symptoms of obstructive sleep apnea?", "image_path": None},

    # Image-relevant questions — paired with a real sample image from each
    # dataset's val split, so the router + classifier actually get exercised.
    {"question": "Could this image be related to Down syndrome, and what are the typical characteristics?",
     "image_path": next((Path(yolo_dirs["down_syndrome"]) / "val").glob("*/*.*"))},

    {"question": "Does this face show signs consistent with autism, and what are common symptoms?",
     "image_path": next((Path(yolo_dirs["autism"]) / "val").glob("*/*.*"))},

    {"question": "Could this image show signs of depression, and what are the typical symptoms?",
     "image_path": next((Path(yolo_dirs["depression"]) / "val").glob("*/*.*"))},

    # A control case: an image attached to a question that ISN'T about any
    # of the 3 supported conditions — should be silently ignored by the router.
    {"question": "What causes migraines?",
     "image_path": next((Path(yolo_dirs["down_syndrome"]) / "val").glob("*/*.*"))},

    {"question": "How does epilepsy differ from a single provoked seizure?", "image_path": None},
]

print(f"{len(eval_questions)} evaluation questions prepared.")

10 evaluation questions prepared.


In [50]:
eval_results = []

for item in eval_questions:

    question = item["question"]
    image_path = item["image_path"]

    # Retrieve text context (used by both text-only and multimodal paths).
    results = retrieve_chunks(question, top_k=5)

    # If an image is attached, first check whether the question
    # is relevant to one of the supported image conditions.
    if image_path is not None:

        routing_result = classify_image(
            image_path,
            question
        )

        # Irrelevant image -> use text-only RAG.
        if routing_result is None:
            prompt = build_rag_prompt(
                question,
                results
            )

        # Relevant image -> use multimodal RAG.
        else:
            prompt = build_multimodal_rag_prompt(
                question,
                results,
                image_path=image_path
            )

    # No image -> use text-only RAG.
    else:
        routing_result = None
        prompt = build_rag_prompt(
            question,
            results
        )

    # Call the LLM.
    response = ollama.chat(
        model="llama3.2:latest",
        messages=[{"role": "user", "content": prompt}],
        options={
            "temperature": 0,
            "num_ctx": 4096
        },
    )

    answer = response["message"]["content"]

    # Collect the retrieved source labels for the table.
    retrieved_sources = [
        f"{meta['book']} (p.{meta['page_start']}-{meta['page_end']})"
        for meta in results["metadatas"][0]
    ]

    eval_results.append({
        "question": question,
        "image_used": image_path.name if image_path else None,
        "routing_result": routing_result,
        "retrieved_sources": retrieved_sources,
        "answer": answer,

        # These 3 fields are filled in manually in Step 3.
        "context_relevant": None,
        "grounded": None,
        "correct": None,
    })

    print(f"Done: {question[:60]}...")

print(f"\n{len(eval_results)} results collected.")

Done: What are the main symptoms of insomnia?...
Done: What are the essential features required to diagnose general...
Done: What are the main causes of disorders of intellectual develo...
Done: What are the main features of attention-deficit hyperactivit...
Done: What are the typical symptoms of obstructive sleep apnea?...

image 1/1 c:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\data\yolo_cls_down_syndrome\val\downSyndrome\down_1002.jpg: 224x224 downSyndrome 1.00, healthy 0.00, 53.7ms
Speed: 30.0ms preprocess, 53.7ms inference, 0.5ms postprocess per image at shape (1, 3, 224, 224)

image 1/1 c:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\data\yolo_cls_down_syndrome\val\downSyndrome\down_1002.jpg: 224x224 downSyndrome 1.00, healthy 0.00, 3.4ms
Speed: 2.0ms preprocess, 3.4ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
Done: Could this image be related

In [51]:
for i, r in enumerate(eval_results):
    print(f"\n{'='*100}")
    print(f"[{i}] Question: {r['question']}")
    if r['image_used']:
        print(f"Image: {r['image_used']}  ->  Routing: {r['routing_result']}")
    print(f"Sources: {r['retrieved_sources']}")
    print(f"\nAnswer:\n{r['answer']}")


[0] Question: What are the main symptoms of insomnia?
Sources: ['Sleep Disorders and Sleep Deprivation (National Academies) (p.2254-2254)', 'Fundamentals of Psychological Disorders (mental use) (p.999-999)', 'Sleep Disorders and Sleep Deprivation (National Academies) (p.2255-2255)', 'Sleep Disorders and Sleep Deprivation (National Academies) (p.2257-2258)', 'Sleep Disorders and Sleep Deprivation (National Academies) (p.2281-2281)']

Answer:
[Source 1] Insomnia is defined by having difficulty falling asleep, maintaining sleep, or by short sleep duration, despite adequate opportunity for a full night’s sleep.

[Source 2] Other insomnia symptoms include daytime consequences, such as tiredness, lack of energy, difficulty concentrating, and/or irritability.

[Source 3] The diagnostic criteria for primary insomnia include: difficulty initiating or maintaining sleep or nonrestorative sleep.

[Source 4] Insomnia is a symptom used with others to diagnose major depression, and the comorbidity, 

In [52]:
eval_results[0].update(context_relevant="Yes", grounded="Yes", correct="Yes")   # Insomnia — clean
eval_results[1].update(context_relevant="Yes", grounded="Yes", correct="Yes")   # GAD — clean
eval_results[2].update(context_relevant="Yes", grounded="Yes", correct="Yes")   # Intellectual disability causes — clean
eval_results[3].update(context_relevant="Yes", grounded="Yes", correct="Yes")   # ADHD — clean
eval_results[4].update(context_relevant="Yes", grounded="Yes", correct="Yes")   # OSA — clean
eval_results[5].update(context_relevant="Yes", grounded="Yes", correct="Yes")   # Down syndrome image — correct classification, real confidence
eval_results[6].update(context_relevant="Yes", grounded="Yes", correct="Yes")   # Autism image — correct classification, real confidence
eval_results[7].update(context_relevant="Yes", grounded="Partial", correct="Partial")  # Depression image — emotion->depression is model inference, not a stated source fact
eval_results[8].update(context_relevant="Yes", grounded="Yes", correct="Yes")   # Migraine + irrelevant image — X%/phantom-classification bug fix confirmed holding
eval_results[9].update(context_relevant="Yes", grounded="Yes", correct="Yes")   # Epilepsy vs. provoked seizure — clean, no topic bleed

In [53]:
import pandas as pd

table_rows = []
for i, r in enumerate(eval_results, start=1):
    table_rows.append({
        "#": i,
        "Question": r["question"][:60] + ("..." if len(r["question"]) > 60 else ""),
        "Retrieved Source(s)": "; ".join(r["retrieved_sources"][:2]) + ("..." if len(r["retrieved_sources"]) > 2 else ""),
        "Context Relevant?": r["context_relevant"],
        "Grounded?": r["grounded"],
        "Correct?": r["correct"],
    })

eval_df = pd.DataFrame(table_rows)
eval_df

,#,Question,Retrieved Source(s),Context Relevant?,Grounded?,Correct?
0,1,What are the main symptoms of insomnia?,Sleep Disorders and Sleep Deprivation (Nationa...,Yes,Yes,Yes
1,2,What are the essential features required to di...,"WHO ICD-11 CDDR (Mental, Behavioural and Neuro...",Yes,Yes,Yes
2,3,What are the main causes of disorders of intel...,"WHO ICD-11 CDDR (Mental, Behavioural and Neuro...",Yes,Yes,Yes
3,4,What are the main features of attention-defici...,"WHO ICD-11 CDDR (Mental, Behavioural and Neuro...",Yes,Yes,Yes
4,5,What are the typical symptoms of obstructive s...,Sleep Disorders and Sleep Deprivation (Nationa...,Yes,Yes,Yes
5,6,"Could this image be related to Down syndrome, ...",Handbook of Neurodevelopmental and Genetic Dis...,Yes,Yes,Yes
6,7,Does this face show signs consistent with auti...,Developmental Screening - CDC (p.1623-1624); H...,Yes,Yes,Yes
7,8,"Could this image show signs of depression, and...","WHO ICD-11 CDDR (Mental, Behavioural and Neuro...",Yes,Partial,Partial
8,9,What causes migraines?,Neurological Disorders - Public Health Challen...,Yes,Yes,Yes
9,10,How does epilepsy differ from a single provoke...,Sleep Disorders and Sleep Deprivation (Nationa...,Yes,Yes,Yes


## 2.7 Export

In [54]:
import json
from pathlib import Path

# Save the configuration needed by the backend
# to load the existing vector store and YOLO models.

export_config = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",

    "chunker": {
        "min_split_tokens": 100,
        "max_split_tokens": 400,
        "window_size": 5,
    },

    "vector_store": {
        "path": "../data/vector_store",
        "collection_name": "neurohealth_chunks",
    },

    "llm_model": "llama3.2:latest",

    "ollama_options": {
        "temperature": 0,
        "num_ctx": 4096,
    },

    "total_chunks": len(final_rag_chunks),

    "yolo_models": {
        "down_syndrome": "runs/classify/runs/down_syndrome_cls-5/weights/best.pt",
        "autism": "runs/classify/runs/autism_cls-4/weights/best.pt",
        "depression": "runs/classify/runs/depression_cls-4/weights/best.pt",
    },
}

CONFIG_PATH = Path("../data/vector_store/config.json")

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(export_config, f, indent=2)

print(f"Config exported to: {CONFIG_PATH}")

Config exported to: ..\data\vector_store\config.json


In [55]:
print(CONFIG_PATH.exists())
print(CONFIG_PATH)

True
..\data\vector_store\config.json


In [56]:
from pathlib import Path

project_root = Path.cwd()

best_models = list(project_root.rglob("best.pt"))

for model in best_models:
    print(model.resolve())

C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\runs\classify\runs\autism_cls\weights\best.pt
C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\runs\classify\runs\autism_cls-2\weights\best.pt
C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\runs\classify\runs\autism_cls-3\weights\best.pt
C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\runs\classify\runs\autism_cls-4\weights\best.pt
C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\runs\classify\runs\autism_cls-5\weights\best.pt
C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth\RAG-Project\notebooks\runs\classify\runs\depression_cls\weights\best.pt
C:\Users\mm\Documents\Junior computer\Training\ITI AI Level 2 Training\NeuroHealth

## Notes on Prompt Engineering & Testing Iterations

During development of the RAG prompt (Section 2.4) and the multimodal fusion prompt (Section 2.5), several issues were identified through iterative testing and are documented here along with their fixes and current status.

### Issue 1 — Cross-topic contamination ("Q8" — epilepsy/stroke)

**Problem:** When asking "What treatment options are available for people with epilepsy?", the generated answer repeatedly included a sentence about aspirin as a stroke treatment:

> "Aspirin is by far the most cost-effective intervention both for treating acute stroke and for preventing a recurrence [Source 2]."

This is factually about a different condition (stroke, not epilepsy) that happened to appear in a retrieved source page discussing multiple neurological topics. The LLM did not recognize the topic mismatch and included it as if it were relevant to epilepsy treatment.

**Mitigation:** A topic-accuracy rule was added to the system prompt instructing the model to verify that each sentence it uses describes the same condition named in the question, and to discard sentences that discuss a different disease even when they come from a source otherwise being used.

**Status:** Reduced in frequency but **not fully eliminated** — this exact sentence reappeared in multiple test runs even with the rule in place, sometimes disappearing and sometimes returning across different runs of the identical question with `temperature=0`. This indicates the fix is a probabilistic mitigation rather than a guarantee, which is a known limitation of prompt-level instructions with a small (3B-parameter) local model. A more complete fix would require retrieval-side changes — finer-grained chunking so multi-topic passages aren't retrieved and presented together.

### Issue 2 — Missing citations

**Problem:** In an early test run, an otherwise well-formed and accurate answer about ADHD symptoms was generated with zero `[Source N]` citations anywhere, despite the system prompt explicitly requiring citation of every claim.

**Cause:** The citation rule was positioned in the middle of a long, multi-section system prompt. The model appeared to deprioritize instructions positioned earlier in a long prompt relative to instructions positioned closer to the generation point.

**Mitigation:** A short, explicit citation reminder was added immediately before the final `Answer:` cue, rather than relying solely on the rule stated earlier in the prompt.

**Status:** Fixed and confirmed — citations appeared consistently in every bullet/sentence across subsequent test batches after this change.

### Issue 3 — Context window truncation

**Problem:** After lengthening the system prompt (adding sectioned rules for grounding, topic accuracy, citations, format, and safety), a test question returned a generic non-answer: *"I'm ready to assist. What is your question about mental, neurodevelopmental, neurological, and sleep disorders?"*

**Cause:** Ollama defaults to a 2048-token context window (`num_ctx`) regardless of the underlying model's actual maximum context length. The combined system prompt + 5 retrieved chunks exceeded this default, causing silent truncation.

**Mitigation:** `num_ctx` was explicitly raised to `8192` in the `options` passed to `ollama.chat`/`ollama.generate`.

**Status:** Fixed and confirmed.

### Issue 4 — Unformatted prompt template (missed `.format()` call)

**Problem:** Immediately after Issue 3's fix, the same fallback ("I'm ready to assist...") reappeared even with `num_ctx` raised.

**Cause:** `build_rag_prompt` was returning the raw `SYSTEM_PROMPT` string containing the literal placeholder text `{question}` and `{context}`, without ever calling `.format(question=question, context=context)` to substitute in the real values. The LLM was receiving the literal text `"{question}"` instead of the actual user question.

**Mitigation:** Added the missing `.format(question=question, context=context)` call at the end of `build_rag_prompt`.

**Status:** Fixed and confirmed.

### Issue 5 — Rigid format template not followed

**Problem:** An earlier version of the prompt required a strict output structure ("Key points:" / "Notes:" headings, max 6 flat bullets). Across a 20-question test batch, **none** of the generated answers actually used these headings — the model continued producing paragraphs, nested bullets, or numbered lists depending on the question, ignoring the structural instruction entirely.

**Cause:** `llama3.2` (3B) reliably follows a small number of hard content constraints but tends to silently drop rigid structural/formatting instructions, especially when they are one of many competing rules in a long prompt.

**Mitigation:** The rigid template was replaced with a content-driven format rule ("choose whichever format best fits the actual content: sentence, flat bullets, or two-level bullets only if the source itself has that structure") rather than forcing a fixed shape.

**Status:** Fixed — output formatting now varies naturally and appropriately with question type (e.g. simple factual questions get short answers, multi-item questions get flat bullets), rather than either being forced into an ignored template or being wildly inconsistent.

### Issue 6 — Placeholder text leaking into multimodal output ("X% confidence")

**Problem:** In the multimodal (image + question) prompt, the classification-result instructions were written as static text containing a literal `X%` placeholder (e.g. `"The image is predicted as healthy (X% confidence)"`) instead of having the real computed confidence value injected at runtime. The model reproduced this literal placeholder verbatim in its answer.

**Mitigation:** The prompt-building function was restructured so the classification-rule text is only constructed at runtime, with the real confidence percentage inserted via an f-string, rather than existing as static template text with a placeholder.

**Status:** Fixed and confirmed — subsequent runs show real computed confidence values (e.g. "100.0% confidence") instead of the literal placeholder.

### Issue 7 — Fabricated classification for irrelevant questions

**Problem:** When a question unrelated to any of the 3 supported conditions (e.g. "What causes migraines?") had an image attached, the router correctly returned `None` (no match), but the generated answer still fabricated a plausible-looking classification statement: *"The image is predicted as healthy (X% confidence). Therefore... the image does not indicate Down syndrome"* — a full hallucinated classification result for a question that never should have triggered one.

**Cause:** The classification-rules section of the prompt was included unconditionally, regardless of whether the router had actually matched a supported condition.

**Mitigation:** The classification-rules block is now only constructed when `classify_image` returns a non-`None` result. When routing returns `None`, the block is omitted entirely and the model is explicitly instructed not to mention any image, classification, or confidence score.

**Status:** Fixed and confirmed — the same migraine question with an attached image now returns a clean text-only answer with zero mention of the image.

### Issue 8 — GPU out-of-memory during batch evaluation

**Problem:** Running the Section 2.6 evaluation loop (10 questions, including 3 image-classification calls) triggered a CUDA out-of-memory error mid-run.

**Cause:** The 6GB laptop GPU was holding the 3 fine-tuned YOLO classification models in memory simultaneously with Ollama loading `llama3.2`, exceeding available VRAM.

**Mitigation:** Passed `keep_alive=0` to `ollama.chat` so the LLM is unloaded from GPU memory immediately after each call, rather than persisting in VRAM across the full evaluation loop.

**Status:** Fixed — the full evaluation batch completed without further memory errors.

### Design limitation (not a bug) — Depression classifier is emotion-based, not diagnosis-based

The depression image dataset is a 7-class facial emotion classifier (Angry, Sad, Happy, Neutral, Fear, Disgust, Surprise), not a binary depressed/not-depressed model. When the LLM connects a detected emotion (e.g. "Angry") to depression relevance, this is model-level inference from general source content, not a directly retrieved fact. This is documented as a scope limitation of the chosen dataset rather than a defect, and is graded as `Partial` rather than fully grounded in the Section 2.6 evaluation table.

### Summary

These issues collectively illustrate a realistic engineering process: two runtime bugs (missing `.format()`, hardcoded `num_ctx`), two prompt-instruction-following limitations specific to using a small local model (`llama3.2` 3B) rather than a larger hosted model (dropped format template, buried citation rule), one recurring probabilistic failure mode that prompt-level rules only partially fix (cross-topic contamination), one multimodal-specific templating bug (unfilled `X%` placeholder and unconditional classification block), and one infrastructure constraint (GPU memory management). All except the cross-topic contamination issue and the depression-mapping design limitation were fully resolved and confirmed fixed through repeated testing.